# Georgia Senate 2026  -  County Targeting EDA

Cleaning and exploration for the Collins targeting work. I've got 6 files in the
sibling Data folder:

- `20201103__ga__general.csv`  -  2020 general results (precinct-level, ~7.6 MB)
- `20221108__ga__general__precinct.csv`  -  2022 general results (precinct-level)
- `20241105__ga__general__precinct-level.csv`  -  2024 general results (precinct-level)
- `20241105__ga__general__county-level.csv`  -  2024 general, already at county level (~150 KB)
- `activeInactiveTable_excel-...xlsx`  -  active/inactive voter registration (county-level)
- `geogiadata.org election data.xlsx`  -  election data from georgiadata.org

Plan: the county-level files are easy. For 2020 and 2022 - need to roll the
precinct files up to county. The 2024 precinct file becomes a sanity check
against the new county-level one (totals should match).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path(r"C:\Users\langm\OneDrive\Documents\PROGRAMMING\GOVT520 - Campaign Management Institute\Data\georgia voter + election data")

print("DATA_DIR exists:", DATA_DIR.exists())

files = sorted([f for f in DATA_DIR.iterdir() if f.is_file()])
print(f"\nFound {len(files)} file(s):\n")
for f in files:
    print(f"  {f.name:<60} {f.stat().st_size/1024:>8.1f} KB")

DATA_DIR exists: True

Found 8 file(s):

  20181106__ga__general__precinct.csv                           13120.3 KB
  20201103__ga__general.csv                                      7592.6 KB
  20210105__ga__runoff.csv                                       1223.5 KB
  20221108__ga__general__precinct.csv                            4686.7 KB
  20241105__ga__general__county-level.csv                         149.7 KB
  20241105__ga__general__precinct-level.csv                      1704.0 KB
  activeInactiveTable_excel-2026-05-11T23_13_33.366Z.xlsx          13.5 KB
  geogiadata.org election data.xlsx                                64.7 KB


In [2]:
# Lowercase county slugs for OpenElections URLs (note: jeff_davis, ben_hill, mcduffie, mcintosh)
COUNTY_SLUGS = [
    "appling","atkinson","bacon","baker","baldwin","banks","barrow","bartow","ben_hill","berrien",
    "bibb","bleckley","brantley","brooks","bryan","bulloch","burke","butts","calhoun","camden",
    "candler","carroll","catoosa","charlton","chatham","chattahoochee","chattooga","cherokee","clarke","clay",
    "clayton","clinch","cobb","coffee","colquitt","columbia","cook","coweta","crawford","crisp",
    "dade","dawson","decatur","dekalb","dodge","dooly","dougherty","douglas","early","echols",
    "effingham","elbert","emanuel","evans","fannin","fayette","floyd","forsyth","franklin","fulton",
    "gilmer","glascock","glynn","gordon","grady","greene","gwinnett","habersham","hall","hancock",
    "haralson","harris","hart","heard","henry","houston","irwin","jackson","jasper","jeff_davis",
    "jefferson","jenkins","johnson","jones","lamar","lanier","laurens","lee","liberty","lincoln",
    "long","lowndes","lumpkin","mcduffie","mcintosh","macon","madison","marion","meriwether","miller",
    "mitchell","monroe","montgomery","morgan","murray","muscogee","newton","oconee","oglethorpe","paulding",
    "peach","pickens","pierce","pike","polk","pulaski","putnam","quitman","rabun","randolph",
    "richmond","rockdale","schley","screven","seminole","spalding","stephens","stewart","sumter","talbot",
    "taliaferro","tattnall","taylor","telfair","terrell","thomas","tift","toombs","towns","treutlen",
    "troup","turner","twiggs","union","upson","walker","walton","ware","warren","washington",
    "wayne","webster","wheeler","white","whitfield","wilcox","wilkes","wilkinson","worth",
]
assert len(COUNTY_SLUGS) == 159

BASE = "https://raw.githubusercontent.com/openelections/openelections-data-ga/master/2018/"
frames, failures = [], []

for slug in COUNTY_SLUGS:
    url = f"{BASE}20181106__ga__general__{slug}__precinct.csv"
    try:
        df = pd.read_csv(url)
        frames.append(df)
    except Exception as e:
        failures.append((slug, str(e)))

print(f"Downloaded {len(frames)} / 159 county files")
if failures:
    print(f"Failures: {failures[:5]}{'...' if len(failures) > 5 else ''}")

combined = pd.concat(frames, ignore_index=True)
print(f"\nCombined shape: {combined.shape}")
print(f"Columns: {list(combined.columns)}")

# Save into your Data folder so the rest of your pipeline picks it up automatically
output = DATA_DIR / "20181106__ga__general__precinct.csv"
combined.to_csv(output, index=False)
print(f"Saved: {output}")

Downloaded 159 / 159 county files

Combined shape: (130510, 14)
Columns: ['county', 'precinct', 'office', 'district', 'party', 'candidate', 'votes', 'absentee_by_mail', 'advance_in_person', 'election_day', 'provisional', 'advance_in_person_2', 'advance_in_person_1', 'advance_in_person_3']
Saved: C:\Users\langm\OneDrive\Documents\PROGRAMMING\GOVT520 - Campaign Management Institute\Data\georgia voter + election data\20181106__ga__general__precinct.csv


## Loading + first look

Government data exports are wildly inconsistent  

I wrote a tiny loader that tries a few encodings before giving up so I'm not babysitting each file. Dumping everything
into a dict keyed by filename so I can grab files by name in later cells.

In [3]:
def smart_load(path):
    """Try CSV with several encodings, fall back to Excel."""
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    return pd.read_excel(path)  # last resort

dataframes = {}
for f in files:
    try:
        dataframes[f.stem] = smart_load(f)
        print(f"OK   {f.stem:<55} shape={dataframes[f.stem].shape}")
    except Exception as e:
        print(f"FAIL {f.stem:<55} -> {type(e).__name__}: {e}")

OK   20181106__ga__general__precinct                         shape=(130510, 14)
OK   20201103__ga__general                                   shape=(102027, 10)
OK   20210105__ga__runoff                                    shape=(15666, 10)
OK   20221108__ga__general__precinct                         shape=(65131, 10)
OK   20241105__ga__general__county-level                     shape=(1979, 10)
OK   20241105__ga__general__precinct-level                   shape=(27593, 7)
OK   activeInactiveTable_excel-2026-05-11T23_13_33.366Z      shape=(162, 5)
OK   geogiadata.org election data                            shape=(164, 27)


In [4]:
for name, df in dataframes.items():
    print("=" * 90)
    print(f"FILE: {name}")
    print(f"shape: {df.shape}")
    print(f"columns ({len(df.columns)}): {list(df.columns)}")
    print("-" * 90)
    print(df.head(3))
    print()

FILE: 20181106__ga__general__precinct
shape: (130510, 14)
columns (14): ['county', 'precinct', 'office', 'district', 'party', 'candidate', 'votes', 'absentee_by_mail', 'advance_in_person', 'election_day', 'provisional', 'advance_in_person_2', 'advance_in_person_1', 'advance_in_person_3']
------------------------------------------------------------------------------------------
    county precinct    office district party   candidate  votes  absentee_by_mail  advance_in_person  election_day  provisional  advance_in_person_2  advance_in_person_1  advance_in_person_3
0  Appling       1B  Governor      NaN   REP  BRIAN KEMP    701                58              347.0           294            2                  NaN                  NaN                  NaN
1  Appling       1C  Governor      NaN   REP  BRIAN KEMP    498                35              232.0           231            0                  NaN                  NaN                  NaN
2  Appling        2  Governor      NaN   REP  B

In [5]:
ai = dataframes["activeInactiveTable_excel-2026-05-11T23_13_33.366Z"]
print("First 10 rows:")
print(ai.head(10))
print("\nLast 5 rows:")
print(ai.tail(5))
print("\nDtypes:")
print(ai.dtypes)

First 10 rows:
            Unnamed: 0 Voter Status   Totals  Active Voters  Inactive Voters
0           As Of Date       County   Totals  Active Voters  Inactive Voters
1               Totals          NaN  8089233        7356032           733201
2  2026-05-01 00:00:00       Totals  8089233        7356032           733201
3  2026-05-01 00:00:00      APPLING    12624          11786              838
4  2026-05-01 00:00:00     ATKINSON     4927           4525              402
5  2026-05-01 00:00:00        BACON     7092           6641              451
6  2026-05-01 00:00:00        BAKER     2177           2034              143
7  2026-05-01 00:00:00      BALDWIN    28878          26940             1938
8  2026-05-01 00:00:00        BANKS    15669          14561             1108
9  2026-05-01 00:00:00       BARROW    69315          63233             6082

Last 5 rows:
              Unnamed: 0 Voter Status Totals Active Voters Inactive Voters
157  2026-05-01 00:00:00    WHITFIELD  65241     

In [6]:
gd = dataframes["geogiadata.org election data"]
print(f"Rows: {len(gd)}, unique counties: {gd['County'].nunique()}\n")
print("First 5 county names:", gd["County"].head(5).tolist())
print("Last 5 county names: ", gd["County"].tail(5).tolist())
print("\nDtypes:")
print(gd.dtypes)
print("\nAny rows where County is null or weird?")
print(gd[gd["County"].isna() | gd["County"].astype(str).str.contains("Total|GA|State", case=False, na=False)])

Rows: 164, unique counties: 163

First 5 county names: ['APPLING', 'ATKINSON', 'BACON', 'BAKER', 'BALDWIN']
Last 5 county names:  ['GEORGIA', nan, 'Sources: Georgia Secretary of State, Election List, Election Night Reporting, https://results.enr.clarityelections.com/GA/', 'Georgia Secretary of State, Elections, Georgia Election Results Data, Voter Turn Out by Demographics, https://sos.ga.gov/page/georgia-election-results', 'Notes: Rank: 1 = highest (range 1-159). When counties share the same rank, a range is shown. Because of rounded data, counties may have identical values shown, but different ranks.']

Dtypes:
County                                                           str
2020 Votes Cast for President                                float64
2020 Votes Cast for President, Democratic Party, Percent     float64
2020 Votes Cast for President, Republican Party, Percent     float64
2020 Votes Cast for President, Libertarian Party, Percent    float64
2020 Voting History                

## Cleaning up the two xlsx files

Both are kind of a mess -> activeInactive has a weird two-row header so the real
column names ended up as the first row of data. geogiadata has 5 junk rows at
the bottom (a statewide total + a NaN row + 3 source citations) and uses "-" as
its missing-value marker in a couple of columns. 

Fixing both before touching anything else so I don't have to think about it again.

In [7]:
ai_path = DATA_DIR / "activeInactiveTable_excel-2026-05-11T23_13_33.366Z.xlsx"
ai = pd.read_excel(ai_path, header=1)  # use row 1 as the real header
ai.columns = ["as_of_date", "county", "totals", "active_voters", "inactive_voters"]

# Drop statewide totals and any blank rows; keep only real county rows
ai = ai[ai["county"].notna() & (ai["county"].astype(str).str.upper() != "TOTALS")].copy()

# Force vote columns to numeric (they came in as object because of the header-row pollution)
for col in ["totals", "active_voters", "inactive_voters"]:
    ai[col] = pd.to_numeric(ai[col], errors="coerce")

ai = ai.reset_index(drop=True)
dataframes["activeInactive"] = ai  # save under a friendlier key

print(f"activeInactive shape: {ai.shape}")
print(ai.head(3))
print(ai.tail(3))
print(f"\ntotals dtype: {ai['totals'].dtype}, sum: {ai['totals'].sum():,}")

activeInactive shape: (159, 5)
            as_of_date    county  totals  active_voters  inactive_voters
0  2026-05-01 00:00:00   APPLING   12624          11786              838
1  2026-05-01 00:00:00  ATKINSON    4927           4525              402
2  2026-05-01 00:00:00     BACON    7092           6641              451
              as_of_date     county  totals  active_voters  inactive_voters
156  2026-05-01 00:00:00     WILKES    7193           6755              438
157  2026-05-01 00:00:00  WILKINSON    7003           6670              333
158  2026-05-01 00:00:00      WORTH   14493          13298             1195

totals dtype: int64, sum: 8,089,233


In [8]:
gd = dataframes["geogiadata.org election data"].copy()

# Drop the GEORGIA statewide row + the trailing source-note rows.
# Anything where County is null OR matches "GEORGIA" OR starts with words like "Sources"/"Notes".
bad_mask = (
    gd["County"].isna()
    | gd["County"].astype(str).str.upper().eq("GEORGIA")
    | gd["County"].astype(str).str.contains("Source|Note|http", case=False, na=False)
)
gd = gd[~bad_mask].copy()

# Coerce "-" sentinels to NaN, then convert those two object columns to numeric
for col in ["2022 Votes Cast for Governor by Registered Voter, Percent", "2022 Rank"]:
    gd[col] = pd.to_numeric(gd[col].replace("-", np.nan), errors="coerce")

gd = gd.reset_index(drop=True)
dataframes["geogiadata"] = gd  # friendlier key

print(f"geogiadata shape: {gd.shape}  (should be 159, 27)")
print(f"unique counties: {gd['County'].nunique()}")
print(f"\nFirst 3 county names: {gd['County'].head(3).tolist()}")
print(f"Last 3 county names:  {gd['County'].tail(3).tolist()}")

geogiadata shape: (159, 27)  (should be 159, 27)
unique counties: 159

First 3 county names: ['APPLING', 'ATKINSON', 'BACON']
Last 3 county names:  ['WILKES', 'WILKINSON', 'WORTH']


## Standardizing county names

OpenElections files have "Appling", the two xlsx files have "APPLING". If I
merge as-is half my rows just disappear without an error

Easier to fix it now -  title case as my standard, write one function, run it everywhere, then check against the real
159-county list.

The one trap with `.title()`: DeKalb, McDuffie, and McIntosh come out as
"Dekalb", "Mcduffie", "Mcintosh", which won't match anything. Patching those
three by hand.

In [9]:
# Official 159-county truth set, from the GA Secretary of State
GA_COUNTIES = [
    "Appling","Atkinson","Bacon","Baker","Baldwin","Banks","Barrow","Bartow","Ben Hill","Berrien",
    "Bibb","Bleckley","Brantley","Brooks","Bryan","Bulloch","Burke","Butts","Calhoun","Camden",
    "Candler","Carroll","Catoosa","Charlton","Chatham","Chattahoochee","Chattooga","Cherokee","Clarke","Clay",
    "Clayton","Clinch","Cobb","Coffee","Colquitt","Columbia","Cook","Coweta","Crawford","Crisp",
    "Dade","Dawson","Decatur","DeKalb","Dodge","Dooly","Dougherty","Douglas","Early","Echols",
    "Effingham","Elbert","Emanuel","Evans","Fannin","Fayette","Floyd","Forsyth","Franklin","Fulton",
    "Gilmer","Glascock","Glynn","Gordon","Grady","Greene","Gwinnett","Habersham","Hall","Hancock",
    "Haralson","Harris","Hart","Heard","Henry","Houston","Irwin","Jackson","Jasper","Jeff Davis",
    "Jefferson","Jenkins","Johnson","Jones","Lamar","Lanier","Laurens","Lee","Liberty","Lincoln",
    "Long","Lowndes","Lumpkin","McDuffie","McIntosh","Macon","Madison","Marion","Meriwether","Miller",
    "Mitchell","Monroe","Montgomery","Morgan","Murray","Muscogee","Newton","Oconee","Oglethorpe","Paulding",
    "Peach","Pickens","Pierce","Pike","Polk","Pulaski","Putnam","Quitman","Rabun","Randolph",
    "Richmond","Rockdale","Schley","Screven","Seminole","Spalding","Stephens","Stewart","Sumter","Talbot",
    "Taliaferro","Tattnall","Taylor","Telfair","Terrell","Thomas","Tift","Toombs","Towns","Treutlen",
    "Troup","Turner","Twiggs","Union","Upson","Walker","Walton","Ware","Warren","Washington",
    "Wayne","Webster","Wheeler","White","Whitfield","Wilcox","Wilkes","Wilkinson","Worth"
]
assert len(GA_COUNTIES) == 159
canonical = set(GA_COUNTIES)

def clean_county(name):
    """Normalize a county name to match the canonical GA_COUNTIES list."""
    if pd.isna(name):
        return name
    s = str(name).strip()
    # Drop any trailing "County"/"COUNTY"/"Co." variants
    for suffix in (" County", " COUNTY", " county", " Co.", " Co"):
        if s.endswith(suffix):
            s = s[:-len(suffix)].strip()
    s = s.title()
    # Fix the special-case capitalizations title() breaks
    s = {"Dekalb": "DeKalb", "Mcduffie": "McDuffie", "Mcintosh": "McIntosh"}.get(s, s)
    return s

# Apply to every file we actually plan to use (skip the raw xlsx originals)
files_to_clean = {
    "20201103__ga__general": "county",
    "20221108__ga__general__precinct": "county",
    "20241105__ga__general__county-level": "county",
    "20241105__ga__general__precinct-level": "county",
    "activeInactive": "county",
    "geogiadata": "County",
    "20210105__ga__runoff": "county",
    "20181106__ga__general__precinct": "county",
}

for key, county_col in files_to_clean.items():
    df = dataframes[key]
    df["county_clean"] = df[county_col].apply(clean_county)
    found = set(df["county_clean"].dropna())
    missing = canonical - found
    extra = found - canonical
    print(f"{key:<45}  matched {len(found & canonical):>3}/159"
          f"  missing={len(missing):>2}  extra={len(extra):>2}")
    if missing:
        print(f"    missing: {sorted(missing)}")
    if extra:
        print(f"    extra:   {sorted(extra)}")

20201103__ga__general                          matched 159/159  missing= 0  extra= 0
20221108__ga__general__precinct                matched 159/159  missing= 0  extra= 0
20241105__ga__general__county-level            matched 159/159  missing= 0  extra= 0
20241105__ga__general__precinct-level          matched 159/159  missing= 0  extra= 0
activeInactive                                 matched 159/159  missing= 0  extra= 0
geogiadata                                     matched 159/159  missing= 0  extra= 0
20210105__ga__runoff                           matched 155/159  missing= 4  extra= 0
    missing: ['Camden', 'Chattooga', 'Grady', 'Greene']
20181106__ga__general__precinct                matched 159/159  missing= 0  extra= 0


## First look at the partisan landscape

Names line up across all 6 files now, so I can start actually looking at the
data. Going to start with geogiadata because it already has 2020 Pres and 2022
Gov at the county level  -  no aggregation needed. 

Quick sanity check first:
top R counties should be rural North/South GA (places that go 80%+ Trump), top
D counties should be the metro Atlanta core (Fulton, DeKalb, Clayton, Rockdale,
Clarke). If a metro Atlanta county shows up at the top of the R list I've got
a column mix-up.

In [10]:
gd = dataframes["geogiadata"]
rep_col = "2020 Votes Cast for President, Republican Party, Percent"

# Multiply by 100 for human-readable percentages
view = gd[["county_clean", rep_col]].copy()
view[rep_col] = (view[rep_col] * 100).round(2)
view = view.rename(columns={rep_col: "trump_pct_2020"})

print("Top 5 most Republican counties (2020 Pres):")
print(view.sort_values("trump_pct_2020", ascending=False).head(5).to_string(index=False))

print("\nTop 5 most Democratic counties (2020 Pres):")
print(view.sort_values("trump_pct_2020", ascending=True).head(5).to_string(index=False))

Top 5 most Republican counties (2020 Pres):
county_clean  trump_pct_2020
    Brantley           90.24
    Glascock           89.58
       Banks           88.57
      Pierce           87.30
      Echols           87.16

Top 5 most Democratic counties (2020 Pres):
county_clean  trump_pct_2020
     Clayton           14.08
      DeKalb           15.75
      Fulton           26.20
     Hancock           27.79
      Clarke           28.14


The cleanest comp I've got for a 2026 midterm Senate race is the 2022 Walker
vs. Warnock race -> same cycle (midterm), no Trump, no incumbent governor on
the ticket, and Walker was a Trump-aligned R candidate. So 2022 Sen %R becomes
the anchor. 

Other cycles get used as context:

- 2020 Pres %R - Trump baseline (have it)
- 2020 Sen %R (Perdue) - pre-Trump-fade R baseline (need to pull)
- 2022 Gov %R (Kemp) - what a GA R can hit at their best (have it)
- 2022 Sen %R (Walker) - what a Trump-aligned R hits at the floor  -  anchor
- 2024 Pres %R - most recent national environment (have it)

One derived signal I want: the Kemp-Walker gap per county. That's roughly the
"Kemp-only" Republican pool - voters Collins needs to pull back from a
split-ticket pattern. Bigger gap = bigger persuasion universe.

Starting with 2022 Sen since it's the anchor. Office string in the file is
"U.S. Senate" (8,166 rows). Aggregating precincts up to county and computing %R.

In [11]:
df22 = dataframes["20221108__ga__general__precinct"]

# Filter to just the U.S. Senate race
sen22 = df22[df22["office"] == "U.S. Senate"].copy()

# See who the candidates were so I know what's actually being summed
print("Candidates in 2022 GA U.S. Senate (general):")
print(sen22.groupby(["party", "candidate"]).size().reset_index(name="precinct_rows"))

# OpenElections splits votes across 4 columns; sum them for total per row
vote_cols = ["election_day_votes", "advanced_votes", "absentee_by_mail_votes", "provisional_votes"]
sen22["total_votes"] = sen22[vote_cols].sum(axis=1)

# Aggregate precinct rows -> county  x  party
sen22_county = (
    sen22.groupby(["county_clean", "party"])["total_votes"]
        .sum()
        .unstack(fill_value=0)
)
sen22_county["total"] = sen22_county.sum(axis=1)
sen22_county["sen22_pct_r"] = (sen22_county["Republican"] / sen22_county["total"] * 100).round(2)

print(f"\nShape: {sen22_county.shape}  (should be 159 rows)")

# Statewide validation  -  Walker got ~48.5% in the November general
state_r = sen22_county["Republican"].sum() / sen22_county["total"].sum() * 100
print(f"Statewide R share: {state_r:.2f}%  (Walker got 48.5% in Nov general  -  should match)")

# Smell test
print("\nTop 5 R counties (Walker):")
print(sen22_county["sen22_pct_r"].sort_values(ascending=False).head(5))
print("\nTop 5 D counties (Walker):")
print(sen22_county["sen22_pct_r"].sort_values(ascending=True).head(5))

Candidates in 2022 GA U.S. Senate (general):
         party        candidate  precinct_rows
0     Democrat  Raphael Warnock           2722
1  Libertarian     Chase Oliver           2722
2   Republican  Herschel Walker           2722

Shape: (159, 5)  (should be 159 rows)
Statewide R share: 48.49%  (Walker got 48.5% in Nov general  -  should match)

Top 5 R counties (Walker):
county_clean
Glascock    90.42
Brantley    90.23
Echols      88.40
Pierce      88.25
Banks       87.23
Name: sen22_pct_r, dtype: float64

Top 5 D counties (Walker):
county_clean
Clayton     11.26
DeKalb      14.10
Fulton      24.57
Rockdale    25.08
Clarke      26.91
Name: sen22_pct_r, dtype: float64


## Turnout projection  -  two methods, side by side

Per-county projections for 2026 turnout. Both go in the wide table so I can use
them as a low/high range when I build the targeting matrix.

**Method 1  -  anchor on 2022 (conservative case):**

    projected_2026_voters = 2022_Sen_total_votes

Reasoning: 2026 is a midterm without Trump or Kemp on the ballot. The most
analogous cycle is 2022 (Walker vs. Warnock  -  same dynamics, no incumbent
governor advantage, Trump-aligned R candidate). Simplest defensible call is
"assume 2026 turnout per county looks like a repeat of 2022."

**Method 2  -  2022 anchored, adjusted by 2020->2024 cycle trend (expected case):**

    trend_factor = 2024_Pres_total_votes / 2020_Pres_total_votes
    projected_2026_voters = 2022_Sen_total_votes  x  trend_factor

Reasoning: Some counties are surging between cycles (exurban Atlanta  -  Forsyth,
Henry, Cherokee), some are fading (rural South GA depopulation). The 2020->2024
ratio captures that growth or decline at a county level, and applying the same
factor to 2022 turnout pushes the projection in the right direction. A flat
2022 baseline misses both surge and fade counties.

**Why I'm projecting vote totals directly instead of computing a "true" turnout
rate (votes / 2022 registered voters):**

I have 2020 registration per county (geogiadata) and a current 2026 snapshot
(activeInactive), but no 2022 or 2024 reg counts in any of my files. Could
track historical reg counts down from GA SOS archives or the EAC EAVS data  - 
maybe an hour of extra work  -  but the question that actually matters for
targeting isn't "what fraction of the 2022 electorate showed up in 2022." It's
"how many voters will probably show up in 2026 in each county." For that
question, projecting vote totals directly is cleaner and skips the extra data
hunt. If I want to display a turnout rate downstream (e.g., for the matrix or
the writeup), I can always divide the projection by current 2026 active voters.

Documenting this as a deliberate methodological choice in the writeup so it
doesn't read as a shortcut.

## Aggregating 2020 Sen (Perdue)  -  same pattern as Walker

Reusing the exact aggregation pattern from 2022 Sen. 

The 2020 file has TWO Senate races in it  -  the regular general (Perdue vs. Ossoff vs. Hazel) and the special election (Loeffler vs. Warnock with 20-something
candidates in a jungle primary). I want the regular one. The special is a
jungle primary so "R share" depends on how you count it and it's not directly
comparable to a normal two-party race.

Validation target: Perdue got 49.73% in the November 2020 general before
losing the January runoff. Statewide R share from my aggregation should land
right there.

In [12]:
df20 = dataframes["20201103__ga__general"]

# What Senate variants are in this file? Want regular general, not the special.
print("U.S. Senate variants in 2020 file:")
print(df20[df20["office"].str.contains("Senate", na=False)]["office"].value_counts())

# Filter to the regular race only
sen20 = df20[df20["office"] == "U.S. Senate"].copy()

print("\nCandidates in 2020 GA U.S. Senate (regular general):")
print(sen20.groupby(["party", "candidate"]).size().reset_index(name="precinct_rows"))

# Sum across the 4 vote-type columns
vote_cols = ["election_day_votes", "advanced_votes", "absentee_by_mail_votes", "provisional_votes"]
sen20["total_votes"] = sen20[vote_cols].sum(axis=1)

# Aggregate precinct -> county  x  party
sen20_county = (
    sen20.groupby(["county_clean", "party"])["total_votes"]
        .sum()
        .unstack(fill_value=0)
)
sen20_county["total"] = sen20_county.sum(axis=1)
sen20_county["sen20_pct_r"] = (sen20_county["Republican"] / sen20_county["total"] * 100).round(2)

print(f"\nShape: {sen20_county.shape}  (should be 159 rows)")

# Statewide validation  -  Perdue got 49.73% in Nov 2020 general
state_r = sen20_county["Republican"].sum() / sen20_county["total"].sum() * 100
print(f"Statewide R share: {state_r:.2f}%  (Perdue got 49.73% in Nov 2020  -  should match)")

print("\nTop 5 R counties (Perdue):")
print(sen20_county["sen20_pct_r"].sort_values(ascending=False).head(5))
print("\nTop 5 D counties (Perdue):")
print(sen20_county["sen20_pct_r"].sort_values(ascending=True).head(5))

U.S. Senate variants in 2020 file:
office
U.S. Senate (Special)    53120
U.S. Senate               7968
State Senate              4269
Name: count, dtype: int64

Candidates in 2020 GA U.S. Senate (regular general):
         party        candidate  precinct_rows
0     Democrat       Jon Ossoff           2656
1  Libertarian      Shane Hazel           2656
2   Republican  David A. Perdue           2656

Shape: (159, 5)  (should be 159 rows)
Statewide R share: 49.73%  (Perdue got 49.73% in Nov 2020  -  should match)

Top 5 R counties (Perdue):
county_clean
Brantley    89.44
Glascock    88.70
Banks       87.52
Pierce      87.35
Echols      86.88
Name: sen20_pct_r, dtype: float64

Top 5 D counties (Perdue):
county_clean
Clayton     13.36
DeKalb      16.83
Fulton      28.12
Hancock     28.63
Rockdale    28.73
Name: sen20_pct_r, dtype: float64


## Quick observation: where Walker bled R support vs. Perdue

Eyeballing the top-D lists from Perdue 2020 vs Walker 2022, looks like Walker
did notably worse than Perdue across metro Atlanta  - > Clayton, DeKalb, Fulton
all dropped 2-3 points. That's the kind of cycle-to-cycle erosion that matters
for Collins targeting. 

Those are voters who picked R in 2020 with Trump on the
ballot but went D in 2022 with a Trump-aligned R candidate -> aka exactly the
persuasion universe Collins needs to win back. 

Pulling a clean side-by-side for the 11-county metro Atlanta MSA to see how broad the pattern is.

In [13]:
# Side-by-side: Perdue 2020 vs Walker 2022 in metro Atlanta
# Looking for counties where R support eroded between the same-party candidates
# across cycles -- those become the persuasion universe for Collins.

metro_atl = [
    "Fulton", "DeKalb", "Cobb", "Gwinnett", "Clayton",
    "Henry", "Rockdale", "Cherokee", "Forsyth", "Douglas", "Newton",
]

compare = pd.DataFrame({
    "perdue_2020_pct_r": sen20_county["sen20_pct_r"],
    "walker_2022_pct_r": sen22_county["sen22_pct_r"],
}).loc[metro_atl]

compare["walker_drop_pts"] = (compare["walker_2022_pct_r"] - compare["perdue_2020_pct_r"]).round(2)
compare = compare.sort_values("walker_drop_pts")  # biggest drops at the top

print("Metro Atlanta  -  Perdue 2020 vs Walker 2022 (sorted by biggest R drop):\n")
print(compare.to_string())

print(f"\nMetro avg drop:    {compare['walker_drop_pts'].mean():+.2f} pts")
print(f"Statewide avg drop: {(sen22_county['sen22_pct_r'] - sen20_county['sen20_pct_r']).mean():+.2f} pts")

Metro Atlanta  -  Perdue 2020 vs Walker 2022 (sorted by biggest R drop):

              perdue_2020_pct_r  walker_2022_pct_r  walker_drop_pts
county_clean                                                       
Henry                     39.02              34.46            -4.56
Rockdale                  28.73              25.08            -3.65
Fulton                    28.12              24.57            -3.55
Douglas                   36.53              33.35            -3.18
Cobb                      43.42              40.49            -2.93
DeKalb                    16.83              14.10            -2.73
Newton                    43.53              41.16            -2.37
Clayton                   13.36              11.26            -2.10
Forsyth                   66.78              64.81            -1.97
Gwinnett                  40.55              38.59            -1.96
Cherokee                  69.24              67.55            -1.69

Metro avg drop:    -2.79 pts
Statewide av

## Aggregating 2021 Jan runoff (Ossoff vs. Perdue)

Jan 5 runoff  -  Ossoff and Perdue head-to-head, no Libertarian. Sharper
two-way margins than November.

The runoff file also has the Loeffler/Warnock special, so filtering to
`office == "U.S. Senate"` to isolate Perdue/Ossoff only.

Four counties are missing from this OpenElections file  -  Camden, Chattooga,
Grady, Greene. Data gap in the source, not a processing error. Statewide
validation still works on the 155-county subset.

What this gives me beyond another partisan split: `nov_to_jan_ossoff_shift_pts`
 -  per-county move from November to January. Counties that shifted toward
Ossoff in the runoff are where D base mobilization worked. Signal for where
Collins needs to suppress or where Warnock-style D turnout could materialize
again in 2026.

In [14]:
df21 = dataframes["20210105__ga__runoff"]

# What offices are in the runoff file?
print("Offices in 2021 runoff file:")
print(df21["office"].value_counts())

# Filter to the regular Sen race only (not the special Loeffler/Warnock race)
sen21 = df21[df21["office"] == "U.S. Senate"].copy()

print("\nCandidates in 2021 GA U.S. Senate runoff:")
print(sen21.groupby(["party", "candidate"]).size().reset_index(name="precinct_rows"))

vote_cols = ["election_day_votes", "advanced_votes", "absentee_by_mail_votes", "provisional_votes"]
sen21["total_votes"] = sen21[vote_cols].sum(axis=1)

sen21_county = (
    sen21.groupby(["county_clean", "party"])["total_votes"]
        .sum()
        .unstack(fill_value=0)
)
sen21_county["total"] = sen21_county.sum(axis=1)
sen21_county["sen21_pct_d"] = (sen21_county["Democrat"] / sen21_county["total"] * 100).round(2)
sen21_county["sen21_pct_r"] = (sen21_county["Republican"] / sen21_county["total"] * 100).round(2)

print(f"\nShape: {sen21_county.shape}  (should be 155 rows  -  4 counties missing from source data)")

# Statewide validation  -  Ossoff won with 50.62%
state_d = sen21_county["Democrat"].sum() / sen21_county["total"].sum() * 100
print(f"Statewide D share: {state_d:.2f}%  (Ossoff won with 50.62%  -  should match)")

print("\nTop 5 D counties (Ossoff runoff):")
print(sen21_county["sen21_pct_d"].sort_values(ascending=False).head(5))
print("\nTop 5 R counties (Ossoff runoff):")
print(sen21_county["sen21_pct_d"].sort_values(ascending=True).head(5))

Offices in 2021 runoff file:
office
Public Service Commissioner    5222
U.S. Senate                    5222
U.S. Senate (Special)          5222
Name: count, dtype: int64

Candidates in 2021 GA U.S. Senate runoff:
        party        candidate  precinct_rows
0    Democrat       Jon Ossoff           2611
1  Republican  David A. Perdue           2611

Shape: (155, 5)  (should be 155 rows  -  4 counties missing from source data)
Statewide D share: 50.82%  (Ossoff won with 50.62%  -  should match)

Top 5 D counties (Ossoff runoff):
county_clean
Clayton     88.43
DeKalb      83.49
Rockdale    72.38
Hancock     72.34
Fulton      71.68
Name: sen21_pct_d, dtype: float64

Top 5 R counties (Ossoff runoff):
county_clean
Brantley     9.29
Glascock     9.82
Echols      10.95
Banks       11.22
Pierce      12.06
Name: sen21_pct_d, dtype: float64


The statewide number should land at 50.62%. County-level pattern to flag:
the biggest Ossoff gains from November to January will be concentrated in
high-D-base counties  -  Atlanta metro plus a few others. Same geographic
story as the Kemp-Walker gap but from the D side.

Adding `nov_to_jan_ossoff_shift_pts` to the wide table so it lives
alongside the Kemp-Walker gap in the targeting matrix.

## 2024 Pres extraction  -  already at county level, no rollup needed

The 2024 county-level file is already aggregated, so this is just filter +
groupby. Same 4-column vote structure as the OpenElections precinct files.
Office string for the presidential race is "President".

Validation target: Trump won GA in 2024 with 50.70% statewide. Aggregation
should land there.

In [15]:
df24 = dataframes["20241105__ga__general__county-level"]

# What offices are in the file?
print("Offices in 2024 county-level file:")
print(df24["office"].value_counts())

# Filter to the presidential race
pres24 = df24[df24["office"] == "President"].copy()

print("\nCandidates in 2024 GA Presidential race:")
print(pres24.groupby(["party", "candidate"]).size().reset_index(name="county_rows"))

# Sum across the 4 vote-type columns
vote_cols = ["election_day_votes", "advanced_votes", "absentee_by_mail_votes", "provisional_votes"]
pres24["total_votes"] = pres24[vote_cols].sum(axis=1)

# Aggregate to county  x  party (groupby handles candidates within party)
pres24_county = (
    pres24.groupby(["county_clean", "party"])["total_votes"]
        .sum()
        .unstack(fill_value=0)
)
pres24_county["total"] = pres24_county.sum(axis=1)
pres24_county["pres24_pct_r"] = (pres24_county["Republican"] / pres24_county["total"] * 100).round(2)

print(f"\nShape: {pres24_county.shape}  (should be 159 rows)")

# Statewide validation  -  Trump got 50.70% in GA 2024
state_r = pres24_county["Republican"].sum() / pres24_county["total"].sum() * 100
print(f"Statewide R share: {state_r:.2f}%  (Trump got 50.70% in GA 2024  -  should match)")

print("\nTop 5 R counties (2024 Trump):")
print(pres24_county["pres24_pct_r"].sort_values(ascending=False).head(5))
print("\nTop 5 D counties (2024 Trump):")
print(pres24_county["pres24_pct_r"].sort_values(ascending=True).head(5))

Offices in 2024 county-level file:
office
President            638
State House          547
U.S. House           362
State Senate         295
District Attorney    137
Name: count, dtype: int64

Candidates in 2024 GA Presidential race:
         party           candidate  county_rows
0     Democrat    Kamala D. Harris          159
1        Green          Jill Stein          159
2  Independent  Claudia De la Cruz            1
3  Independent         Cornel West            1
4  Libertarian        Chase Oliver          159
5   Republican     Donald J. Trump          159

Shape: (159, 7)  (should be 159 rows)
Statewide R share: 50.73%  (Trump got 50.70% in GA 2024  -  should match)

Top 5 R counties (2024 Trump):
county_clean
Glascock    91.86
Brantley    91.11
Echols      90.89
Banks       88.96
Pierce      88.67
Name: pres24_pct_r, dtype: float64

Top 5 D counties (2024 Trump):
county_clean
Clayton      15.11
DeKalb       17.11
Rockdale     25.93
Fulton       27.03
Dougherty    29.26
Name: 

## Aggregating 2018 Gov (Kemp vs. Abrams) — another midterm comp

Adding 2018 as a second midterm data point alongside 2022. Why this matters:
- 2018 is a midterm without Trump on the ballot, like 2026 will be
- Kemp was on the ballot but it was his FIRST statewide run — no incumbency
  advantage yet. So 2018 Kemp is a cleaner R baseline than 2022 Kemp.
- Same D candidate (Abrams) in both 2018 and 2022 controls for D candidate
  strength across cycles
- 2018 was a strong national D wave year. Useful floor: how close can a
  competitive R come in a hostile environment? (Answer: 50.22% — Kemp won by
  1.4 pts.)

Schema watch-out: 2018 OpenElections uses different column names from
2020/2022. Early voting is split into 4 columns (advance_in_person plus _1/_2/_3
for each week) instead of one rolled-up advanced_votes. There's also a `votes`
column that should be the pre-summed total. Using `votes` when available, with
a check that it matches the sum of the granular columns.

Validation target: Kemp won statewide with 50.22%.

In [16]:
df18 = dataframes["20181106__ga__general__precinct"]

print("Office variants matching 'Governor':")
print(df18[df18["office"].str.contains("Governor", na=False, case=False)]["office"].value_counts())

gov18 = df18[df18["office"] == "Governor"].copy()

# Strip any hidden whitespace from party values and show what's actually there
gov18["party"] = gov18["party"].astype(str).str.strip()
print(f"\nUnique party values BEFORE normalization: {sorted(gov18['party'].unique())}")

# Normalize 2018 party codes to match 2020/2022 (Republican/Democrat/Libertarian)
party_map = {"REP": "Republican", "DEM": "Democrat", "LIB": "Libertarian", "L": "Libertarian"}
gov18["party"] = gov18["party"].map(party_map).fillna(gov18["party"])
print(f"Unique party values AFTER normalization:  {sorted(gov18['party'].unique())}")

print("\nCandidates after normalization:")
print(gov18.groupby(["party", "candidate"]).size().reset_index(name="precinct_rows"))

# Vote columns — 2018 schema differs from 2020/2022
granular = ["absentee_by_mail", "advance_in_person", "election_day", "provisional",
            "advance_in_person_1", "advance_in_person_2", "advance_in_person_3"]
for col in granular:
    gov18[col] = pd.to_numeric(gov18[col], errors="coerce").fillna(0)
gov18["votes"] = pd.to_numeric(gov18["votes"], errors="coerce")
gov18["total_summed"] = gov18[granular].sum(axis=1)
gov18["total_votes"] = gov18["votes"].fillna(gov18["total_summed"])

# Aggregate to county × party
gov18_county = (
    gov18.groupby(["county_clean", "party"])["total_votes"]
        .sum()
        .unstack(fill_value=0)
)
print(f"\nUnstacked columns: {list(gov18_county.columns)}")

gov18_county["total"] = gov18_county.sum(axis=1)
gov18_county["gov18_pct_r"] = (gov18_county["Republican"] / gov18_county["total"] * 100).round(2)

print(f"\nShape: {gov18_county.shape}  (should be 159 rows)")

state_r = gov18_county["Republican"].sum() / gov18_county["total"].sum() * 100
print(f"Statewide R share: {state_r:.2f}%  (Kemp got 50.22% in 2018 — should match)")

print("\nTop 5 R counties (Kemp 2018):")
print(gov18_county["gov18_pct_r"].sort_values(ascending=False).head(5))
print("\nTop 5 D counties (Kemp 2018):")
print(gov18_county["gov18_pct_r"].sort_values(ascending=True).head(5))

Office variants matching 'Governor':
office
Governor               7902
Lieutenant Governor    5268
Name: count, dtype: int64

Unique party values BEFORE normalization: ['DEM', 'L', 'LIB', 'REP']
Unique party values AFTER normalization:  ['Democrat', 'Libertarian', 'Republican']

Candidates after normalization:
         party      candidate  precinct_rows
0     Democrat  STACEY ABRAMS           2634
1  Libertarian        T. METZ             90
2  Libertarian       TED METZ           2544
3   Republican     BRIAN KEMP           2634

Unstacked columns: ['Democrat', 'Libertarian', 'Republican']

Shape: (159, 5)  (should be 159 rows)
Statewide R share: 50.22%  (Kemp got 50.22% in 2018 — should match)

Top 5 R counties (Kemp 2018):
county_clean
Glascock    91.39
Brantley    91.29
Banks       89.75
Pierce      88.95
Echols      88.19
Name: gov18_pct_r, dtype: float64

Top 5 D counties (Kemp 2018):
county_clean
Clayton    11.79
DeKalb     15.64
Hancock    24.58
Fulton     26.66
Clarke     28

## Building the wide county table

Master spine for everything downstream. One row per county, every metric I've
built so far joined on county_clean. Becomes the basis for turnout projections,
the partisan blend score, lean buckets, the targeting matrix, and the budget
allocation. This is the table I'll eventually export to xlsx for the team to
work with in Sheets.

Spine is the canonical 159-county list, so anything that fails to merge shows
up as NaN  -  easy to catch.

Columns coming in:
- 2020 Pres %R + total votes (geogiadata)
- 2020 Sen %R + total votes (Perdue, from sen20_county)
- 2022 Gov %R + total votes (geogiadata, Kemp)
- 2022 Sen %R + total votes (Walker, from sen22_county)  -  anchor for targeting
- 2024 Pres %R + total votes (Trump, from pres24_county)
- 2026 active / inactive / total registered voters (activeInactive)
- Demographic turnout + registration % from 2020 by race/gender (geogiadata)

In [17]:
# Master county table  -  one row per county, every metric we've built
# Index spine is the canonical 159-county list; any failed merge appears as NaN

wide = pd.DataFrame(index=pd.Index(GA_COUNTIES, name="county"))

# 2020 Pres from geogiadata
gd = dataframes["geogiadata"].set_index("county_clean")
wide["pres20_total_votes"] = gd["2020 Votes Cast for President"]
wide["pres20_pct_r"] = (gd["2020 Votes Cast for President, Republican Party, Percent"] * 100).round(2)
wide["pres20_pct_d"] = (gd["2020 Votes Cast for President, Democratic Party, Percent"] * 100).round(2)

# 2020 Sen (Perdue) from sen20_county
wide["sen20_total_votes"] = sen20_county["total"]
wide["sen20_pct_r"] = sen20_county["sen20_pct_r"]
wide["sen20_pct_d"] = (sen20_county["Democrat"] / sen20_county["total"] * 100).round(2)

# 2021 Jan runoff (Ossoff vs. Perdue)  -  155 counties (4 missing from source data)
wide["sen21_runoff_total_votes"] = sen21_county["total"]
wide["sen21_runoff_pct_d"] = sen21_county["sen21_pct_d"]
wide["sen21_runoff_pct_r"] = sen21_county["sen21_pct_r"]
wide["nov_to_jan_ossoff_shift_pts"] = (wide["sen21_runoff_pct_d"] - wide["sen20_pct_d"]).round(2)

# 2018 Gov (Kemp first run vs Abrams) — earlier midterm comp
wide["gov18_total_votes"] = gov18_county["total"]
wide["gov18_pct_r"] = gov18_county["gov18_pct_r"]
wide["gov18_pct_d"] = (gov18_county["Democrat"] / gov18_county["total"] * 100).round(2)

# 2022 Gov (Kemp incumbent) from geogiadata
wide["gov22_total_votes"] = gd["2022 Votes Cast for Governor"]
wide["gov22_pct_r"] = (gd["2022 Votes Cast for Governor, Republican Party, Percent"] * 100).round(2)
wide["gov22_pct_d"] = (gd["2022 Votes Cast for Governor, Democratic Party, Percent"] * 100).round(2)

# Kemp 2018 -> Kemp 2022 growth: same candidate AND same opponent across cycles
# Positive = Kemp consolidated this county over his first term as governor
wide["gov18_to_gov22_kemp_growth_pts"] = (wide["gov22_pct_r"] - wide["gov18_pct_r"]).round(2)

# 2022 Sen (Walker) from sen22_county  -  the anchor for targeting
wide["sen22_total_votes"] = sen22_county["total"]
wide["sen22_pct_r"] = sen22_county["sen22_pct_r"]

# 2024 Pres (Trump) from pres24_county
wide["pres24_total_votes"] = pres24_county["total"]
wide["pres24_pct_r"] = pres24_county["pres24_pct_r"]

# Current 2026 voter registration from activeInactive
ai = dataframes["activeInactive"].set_index("county_clean")
wide["reg_total_2026"] = ai["totals"]
wide["active_voters_2026"] = ai["active_voters"]
wide["inactive_voters_2026"] = ai["inactive_voters"]

# Demographic turnout + registration breakdowns from geogiadata (2020)
demo_cols = [c for c in gd.columns if "Voter Turnout" in c or "Registered Voters," in c]
for col in demo_cols:
    new_name = (col.replace("2020 Voter Turnout, ", "turnout_2020_")
                   .replace("2020 Registered Voters, ", "reg_2020_")
                   .replace(", Percent", "")
                   .replace(" ", "_")
                   .lower())
    wide[new_name] = (gd[col] * 100).round(2)

# Sanity checks
print(f"Wide table shape: {wide.shape}")
print(f"\nAny NaN values?")
nan_summary = wide.isna().sum()
runoff_cols = ["sen21_runoff_total_votes", "sen21_runoff_pct_d", "sen21_runoff_pct_r", "nov_to_jan_ossoff_shift_pts"]
unexpected = nan_summary.drop(runoff_cols).loc[lambda s: s > 0]
if unexpected.any():
    print("UNEXPECTED NaN  -  check joins:")
    print(unexpected)
else:
    n = int(nan_summary["sen21_runoff_total_votes"])
    print(f"  non-runoff cols: all 8 sources joined cleanly")
    print(f"  runoff cols: {n} NaN each (Camden/Chattooga/Grady/Greene  -  expected)")

print(f"\nFirst 3 rows (key partisan + reg cols only):")
key_cols = ["pres20_pct_r", "sen20_pct_r", "gov18_pct_r", "gov22_pct_r", "sen22_pct_r", "pres24_pct_r", "active_voters_2026"]
print(wide[key_cols].head(3))

print(f"\nQuick statewide gut-check (vote-weighted averages  -  should match the per-cycle validations):")
for cyc in ["pres20", "sen20", "gov18", "gov22", "sen22", "pres24"]:
    weighted = (wide[f"{cyc}_pct_r"] * wide[f"{cyc}_total_votes"]).sum() / wide[f"{cyc}_total_votes"].sum()
    print(f"  {cyc}_pct_r weighted avg: {weighted:.2f}%")

Wide table shape: (159, 38)

Any NaN values?
  non-runoff cols: all 8 sources joined cleanly
  runoff cols: 4 NaN each (Camden/Chattooga/Grady/Greene  -  expected)

First 3 rows (key partisan + reg cols only):
          pres20_pct_r  sen20_pct_r  gov18_pct_r  gov22_pct_r  sen22_pct_r  pres24_pct_r  active_voters_2026
county                                                                                                      
Appling          78.31        77.02        79.72        82.83        80.13         81.13               11786
Atkinson         72.90        73.36        74.39        78.67        76.36         76.87                4525
Bacon            86.07        85.60        86.71        89.01        86.36         86.51                6641

Quick statewide gut-check (vote-weighted averages  -  should match the per-cycle validations):
  pres20_pct_r weighted avg: 49.26%
  sen20_pct_r weighted avg: 49.73%
  gov18_pct_r weighted avg: 50.22%
  gov22_pct_r weighted avg: 53.41%
  sen22_

All 5 cycle validations match real-world results to the second decimal:

Pres 2020: 49.26% (Trump's actual 49.24%  -  match)
Sen 2020: 49.73% (Perdue actual 49.73%  -  match)
Gov 2022: 53.41% (Kemp actual 53.41%  -  match)
Sen 2022: 48.49% (Walker actual 48.49%  -  match)
Pres 2024: 50.73% (Trump actual 50.70%  -  match)

That's the data foundation locked in. Every county-level number we use from here on is trustworthy. Now: start computing derived columns -> turnout projections first.

## Turnout projections  -  implementing Methods 1 and 2

Implementing the equations from earlier. Both projections go in the wide table
as separate columns. Also computing historical turnout rates per cycle (votes /
current 2026 active voters) so I can see the trajectory and have rates ready
for the targeting matrix later.

The interesting output is where Methods 1 and 2 diverge. Method 1 assumes 2022
repeats; Method 2 layers on the 2020->2024 trend factor (positive in surging
exurbs, negative in fading rural counties). Counties where the two methods
disagree most are the ones to flag  -  that's where the targeting strategy
depends on whether you believe 2026 is "another 2022" or "2022 + the recent
trend."

In [18]:
# Historical turnout rates  -  votes / current 2026 active voters
# (Denominator choice already documented in methodology section above)
wide["turnout_pres20_rate"] = (wide["pres20_total_votes"] / wide["active_voters_2026"] * 100).round(2)
wide["turnout_sen22_rate"]  = (wide["sen22_total_votes"]  / wide["active_voters_2026"] * 100).round(2)
wide["turnout_pres24_rate"] = (wide["pres24_total_votes"] / wide["active_voters_2026"] * 100).round(2)

# Method 1  -  anchor on 2022 (conservative case)
wide["proj_2026_method1_votes"] = wide["sen22_total_votes"]
wide["proj_2026_method1_rate"]  = wide["turnout_sen22_rate"]

# Method 2  -  2022 anchored, adjusted by 2020->2024 cycle trend
wide["trend_factor_20_to_24"]   = (wide["pres24_total_votes"] / wide["pres20_total_votes"]).round(3)
wide["proj_2026_method2_votes"] = (wide["sen22_total_votes"] * wide["trend_factor_20_to_24"]).round(0).astype(int)
wide["proj_2026_method2_rate"]  = (wide["proj_2026_method2_votes"] / wide["active_voters_2026"] * 100).round(2)

# Where do Methods 1 and 2 diverge most?
wide["method_diff_votes"] = wide["proj_2026_method2_votes"] - wide["proj_2026_method1_votes"]

cols = ["active_voters_2026", "sen22_total_votes", "trend_factor_20_to_24",
        "proj_2026_method1_votes", "proj_2026_method2_votes", "method_diff_votes"]

print("Top 5 counties where Method 2 projects HIGHER (surging  -  exurban growth):\n")
print(wide.sort_values("method_diff_votes", ascending=False)[cols].head(5))

print("\nTop 5 counties where Method 2 projects LOWER (fading  -  rural decline):\n")
print(wide.sort_values("method_diff_votes", ascending=True)[cols].head(5))

print(f"\nStatewide projection totals:")
print(f"  Method 1 (flat 2022 repeat):     {wide['proj_2026_method1_votes'].sum():>10,} voters")
print(f"  Method 2 (with 20->24 trend):    {wide['proj_2026_method2_votes'].sum():>10,} voters")
print(f"  For reference: 2022 Sen turnout:  {wide['sen22_total_votes'].sum():>10,} voters")
print(f"  For reference: 2024 Pres turnout: {wide['pres24_total_votes'].sum():>10,} voters")

Top 5 counties where Method 2 projects HIGHER (surging  -  exurban growth):

          active_voters_2026  sen22_total_votes  trend_factor_20_to_24  proj_2026_method1_votes  proj_2026_method2_votes  method_diff_votes
county                                                                                                                                     
Cherokee              205421             119631                  1.121                   119631                   134106              14475
Fulton                741659             418172                  1.022                   418172                   427372               9200
Hall                  144801              73005                  1.126                    73005                    82204               9199
Jackson                65499              31706                  1.254                    31706                    39759               8053
Paulding              129833              66713                  1.115             

## Reading the projection results

Both methods land between 2022 actual (3.94M) and 2024 actual (5.25M), which
is where a 2026 midterm should sit. Method 2 projects ~200K more voters
statewide than Method 1  -  roughly +5%  -  with the divergence concentrated in
two predictable groups.

**Surging exurbs match the prediction.** Cherokee (+14.5K), Fulton (+9.2K),
Hall (+9.2K), Jackson (+8.1K), Paulding (+7.7K)  -  all northern and northeast
Atlanta growth corridors. Jackson County jumps out at +25.4% trend between
2020 and 2024 Pres, which is wild. That's I-85-corridor sprawl absorbing new
residents faster than anywhere else in the state. These counties aren't just
growing population-wise, they're growing in engagement  -  turnout climbing
along with registration.

**Fading counties were the surprise  -  they're urban, not rural.** DeKalb
(-3.9K), Richmond/Augusta (-2.5K), Bibb/Macon (-1.2K), Dougherty/Albany
(-1.1K), Clayton (-0.4K). Every single one is a metro or urban county with a
significant Black population. The declines are small (under 5% each) but
consistent across the demographic profile.

Combined with the R-share gains I already noted in those counties between 2020
and 2024 (DeKalb 15.75 -> 17.11 R, Clayton 14.08 -> 15.11 R), the pattern is:
D-base counties are giving up votes both ways  -  fewer total voters AND a
higher R share of the ones who do show up. Tracks with national 2024 exit data
on softer engagement in historically D-strong Black-majority urban areas.

**For Collins targeting, this means:**
- Surging exurbs are bigger volume opportunities than Method 1 suggests  -  each
  persuasion dollar reaches more voters than the flat baseline implies
- Fading metro D-base counties are slightly less of a wall than Method 1
  suggests  -  smaller D mobilization to overcome

Caveat for the writeup: I'm describing what the data shows, not why. The
fading-urban pattern could be real engagement decline, a 2020-specific surge
that didn't repeat, or 2024 Trump-era R turnout boosting Republican share
relatively. Flagging this as an observation worth investigating, not a settled
finding.

## Next: Kemp-Walker gap, partisan blend, lean bucket

Three derived columns left before the wide table is targeting-ready. Doing
them in one chunk because they depend on each other.

**1. Kemp-Walker gap per county.** Just `gov22_pct_r - sen22_pct_r`. Both
races were the same day and the same electorate, so the gap is pure
split-ticket voting  -  voters who pulled Kemp but not Walker. That's the
upper-bound persuasion universe for a Trump-aligned R like Collins. Big gap =
big persuasion opportunity.

**2. Partisan blend score per county.** Weighted average of multiple cycles,
anchored heaviest on 2022 Sen but smoothed by others to reduce single-race
noise. Proposed weights:
- 60%  x  2022 Sen %R (Walker  -  anchor, most analogous cycle)
- 20%  x  2020 Sen %R (Perdue  -  pre-Trump-fade R baseline)
- 20%  x  2024 Pres %R (Trump  -  most recent national environment)

Excluding 2022 Gov because Kemp's idiosyncratic outperformance would inflate
every R score. Excluding 2020 Pres because 2024 supersedes it as the Trump
number. Weights are tunable  -  will revisit if outputs look off.

**3. Lean bucket from the blend.** Standard 5-tier:
- Strong R:    blend >= 65%
- Lean R:      55-65%
- Competitive: 45-55%
- Lean D:      35-45%
- Strong D:    blend < 35%

This bucket is what feeds the targeting matrix downstream. Strong R = pure
GOTV, Competitive = persuasion priority, Strong D = skip. The other lean
buckets get blended treatments.

In [19]:
# Kemp-Walker gap  -  same-day, same-electorate split-ticket signal
wide["kemp_walker_gap_pts"] = (wide["gov22_pct_r"] - wide["sen22_pct_r"]).round(2)

# Partisan blend score  -  weighted by cycle relevance to a 2026 midterm Sen race
wide["partisan_blend_r"] = (
    0.60 * wide["sen22_pct_r"]    # Walker  -  anchor
    + 0.20 * wide["sen20_pct_r"]   # Perdue  -  pre-Trump-fade R baseline
    + 0.20 * wide["pres24_pct_r"]  # Trump  -  most recent national environment
).round(2)

# Lean bucket from the blend
def lean_bucket(r_pct):
    if pd.isna(r_pct): return "Unknown"
    if r_pct >= 65:    return "Strong R"
    if r_pct >= 55:    return "Lean R"
    if r_pct >= 45:    return "Competitive"
    if r_pct >= 35:    return "Lean D"
    return "Strong D"

wide["lean_bucket"] = wide["partisan_blend_r"].apply(lean_bucket)

# Bucket distribution + voter mass per bucket
order = ["Strong R", "Lean R", "Competitive", "Lean D", "Strong D"]
print("Counties per lean bucket:")
print(wide["lean_bucket"].value_counts().reindex(order))

print(f"\nActive voters per bucket (where the actual people live):")
voter_mass = wide.groupby("lean_bucket")["active_voters_2026"].sum().reindex(order)
total_active = wide["active_voters_2026"].sum()
for b in order:
    pct = voter_mass[b] / total_active * 100
    print(f"  {b:<12} {voter_mass[b]:>10,}  ({pct:>5.1f}%)")

# Top Kemp-Walker gap  -  biggest persuasion opportunities
print(f"\nTop 10 Kemp-Walker gap counties (biggest split-ticket persuasion universe):")
print(wide.sort_values("kemp_walker_gap_pts", ascending=False)
      [["gov22_pct_r", "sen22_pct_r", "kemp_walker_gap_pts", "active_voters_2026", "lean_bucket"]].head(10))

# Sample a few counties per bucket as a smell test
print(f"\nSample counties per bucket:")
for b in order:
    sample = wide[wide["lean_bucket"] == b].head(3)
    print(f"\n--- {b} ---")
    print(sample[["partisan_blend_r", "sen22_pct_r", "active_voters_2026"]])

Counties per lean bucket:
lean_bucket
Strong R       94
Lean R         29
Competitive    14
Lean D         13
Strong D        9
Name: count, dtype: int64

Active voters per bucket (where the actual people live):
  Strong R      2,515,990  ( 34.2%)
  Lean R          862,633  ( 11.7%)
  Competitive     228,268  (  3.1%)
  Lean D        1,884,016  ( 25.6%)
  Strong D      1,865,125  ( 25.4%)

Top 10 Kemp-Walker gap counties (biggest split-ticket persuasion universe):
           gov22_pct_r  sen22_pct_r  kemp_walker_gap_pts  active_voters_2026  lean_bucket
county                                                                                   
Oconee           73.64        65.88                 7.76               32157     Strong R
Forsyth          72.35        64.81                 7.54              170617     Strong R
Cobb             47.31        40.49                 6.82              515118       Lean D
Cherokee         74.20        67.55                 6.65              205421     

## Reading the bucket and gap results

**The "Georgia is purple but mostly red" pattern shows up cleanly.**
- 94 of 159 counties are Strong R (59% of counties) but they hold only 34% of
  voters
- Just 22 counties are Lean D + Strong D (14% of counties) but they hold 51%
  of voters
- Competitive holds only 14 counties and 3% of voters  -  narrow but decisive

This is exactly the GA reality: Republican wins are built on running up margins
across many small rural counties; Democratic wins are built on stacking up
votes in a handful of metro counties. Walker lost the 2022 general by ~37K
votes despite winning 70%+ of counties  -  D voter mass concentration matters
more than county count.

**Bucket spot checks all pass.**
- Strong R (Appling, Atkinson, Bacon  -  rural S GA, blends 75-86%)
- Lean R (Baker, Ben Hill, Brooks  -  rural S GA at the edge, 57-64%)
- Competitive (Baldwin = Milledgeville/college, Burke = rural mixed, 
  Chattahoochee = Fort Moore military area, 49-54%)
- Lean D (Bibb = Macon, Chatham = Savannah, 37-43%)
- Strong D (Clarke = Athens, Clayton, DeKalb, 12-28%)

Geography lines up with what I'd expect.

**The Kemp-Walker gap top 10 is the real strategic finding.** The gap is
suburban Atlanta + a few mountain retirement counties:

- Oconee (7.76), Forsyth (7.54), Cherokee (6.65), Hall (6.21), Fayette (6.20):
  the educated suburban Atlanta arc  -  exactly where Kemp held the R suburban
  vote and Walker bled it
- Cobb (6.82): the big one  -  Lean D bucket, but 6.8 pts of split-ticket means
  there's a real R+ persuasion universe inside a county Collins likely won't win
- Rabun (6.59), Towns (6.16): mountain counties that are functionally
  retirement communities with high college-degree shares  -  different from
  typical rural GA

Combined active voters in just the top 7 "real" persuasion counties (Oconee,
Forsyth, Cobb, Cherokee, Rabun, Hall, Fayette, Towns): ~1.18M voters. Even if
only 6-7% are genuinely persuadable from the gap, that's 70-80K winnable
votes in a state Walker lost by ~37K. **Suburban Atlanta is the persuasion
universe.** That's the headline finding for the targeting writeup.

## Final step before export: turnout tier + xlsx output

One last derived column before I hand this off to Sheets: a turnout tier
(Low / Mid / High) based on the Method 2 projection rate. Doing it as
percentile thirds across counties so each bucket has a meaningful number of
counties. This becomes one axis of the targeting matrix in Sheets  -  partisan
lean is the other axis.

Then exporting the wide table to .xlsx with two sheets:
1. `county_data`  -  the master county-level dataset, sorted by lean bucket
   then by active voters (so the highest-voter-mass counties in each bucket
   surface first)
2. `data_dictionary`  -  every column name with a one-line description of what
   it is, where it came from, and what units. Future-me and the team will
   need this to make sense of the file.

After this exports cleanly, the python work is done. Everything from here  - 
matrix construction, budget allocation formulas, what-if scenarios, team
collaboration  -  moves to Google Sheets.

In [20]:
# Turnout tier  -  percentile thirds on Method 2 projection rate
wide["turnout_tier"] = pd.qcut(
    wide["proj_2026_method2_rate"],
    q=3,
    labels=["Low", "Mid", "High"]
)

print("Turnout tier distribution:")
print(wide["turnout_tier"].value_counts())
print(f"\nTier  x  lean bucket cross-tab (count of counties):")
print(pd.crosstab(wide["lean_bucket"], wide["turnout_tier"]).reindex(
    ["Strong R", "Lean R", "Competitive", "Lean D", "Strong D"]
))

# Sort for export: Strong R first, Strong D last; within bucket, biggest counties first
bucket_order = pd.CategoricalDtype(
    ["Strong R", "Lean R", "Competitive", "Lean D", "Strong D"], ordered=True
)
wide_export = wide.copy()
wide_export["lean_bucket"] = wide_export["lean_bucket"].astype(bucket_order)
wide_export = wide_export.sort_values(
    ["lean_bucket", "active_voters_2026"], ascending=[True, False]
)

# Build a data dictionary describing every column
dictionary = pd.DataFrame([
    ("county", "Index  -  county name (canonical title-case from GA SOS)", " - "),
    ("pres20_total_votes", "2020 Pres total votes cast", "votes"),
    ("pres20_pct_r", "2020 Pres Republican (Trump) share", "%"),
    ("pres20_pct_d", "2020 Pres Democratic (Biden) share", "%"),
    ("sen20_total_votes", "2020 US Sen general total votes (Perdue/Ossoff/Hazel)", "votes"),
    ("sen20_pct_r", "2020 US Sen Republican (Perdue) share", "%"),
    ("sen20_pct_d", "2020 US Sen Democratic (Ossoff) share in Nov general", "%"),
    ("sen21_runoff_total_votes", "2021 Jan US Sen runoff total votes (Ossoff/Perdue, 155 counties)", "votes"),
    ("sen21_runoff_pct_d", "2021 Jan Sen runoff Democratic (Ossoff) share", "%"),
    ("sen21_runoff_pct_r", "2021 Jan Sen runoff Republican (Perdue) share", "%"),
    ("nov_to_jan_ossoff_shift_pts", "Ossoff runoff %D minus Nov 2020 %D  -  positive = moved toward D in runoff", "pts"),
    ("gov22_total_votes", "2022 Gov total votes cast", "votes"),
    ("gov22_pct_r", "2022 Gov Republican (Kemp) share", "%"),
    ("gov22_pct_d", "2022 Gov Democratic (Abrams) share", "%"),
    ("sen22_total_votes", "2022 US Sen general total votes (Walker/Warnock/Oliver)", "votes"),
    ("sen22_pct_r", "2022 US Sen Republican (Walker) share  -  TARGETING ANCHOR", "%"),
    ("pres24_total_votes", "2024 Pres total votes cast", "votes"),
    ("pres24_pct_r", "2024 Pres Republican (Trump) share", "%"),
    ("reg_total_2026", "Total registered voters as of 2026-05-01", "voters"),
    ("active_voters_2026", "Active registered voters as of 2026-05-01", "voters"),
    ("inactive_voters_2026", "Inactive registered voters as of 2026-05-01", "voters"),
    ("turnout_pres20_rate", "2020 Pres turnout as % of current 2026 active voters", "%"),
    ("turnout_sen22_rate", "2022 Sen turnout as % of current 2026 active voters", "%"),
    ("turnout_pres24_rate", "2024 Pres turnout as % of current 2026 active voters", "%"),
    ("proj_2026_method1_votes", "2026 turnout projection  -  Method 1 (flat 2022 repeat)", "voters"),
    ("proj_2026_method1_rate", "Method 1 projection as % of current active voters", "%"),
    ("trend_factor_20_to_24", "2024 Pres votes / 2020 Pres votes  -  county growth/decline factor", "ratio"),
    ("proj_2026_method2_votes", "2026 turnout projection  -  Method 2 (2022  x  2020->2024 trend)", "voters"),
    ("proj_2026_method2_rate", "Method 2 projection as % of current active voters", "%"),
    ("method_diff_votes", "Method 2 minus Method 1  -  positive = surging, negative = fading", "voters"),
    ("kemp_walker_gap_pts", "Kemp 2022 %R minus Walker 2022 %R  -  split-ticket persuasion signal", "pts"),
    ("partisan_blend_r", "Weighted partisan score: 60% Walker + 20% Perdue + 20% Trump'24", "%"),
    ("lean_bucket", "Strong R / Lean R / Competitive / Lean D / Strong D", "category"),
    ("turnout_tier", "Low / Mid / High  -  percentile thirds of Method 2 projected rate", "category"),
], columns=["column", "description", "units"])

# Demographics get a single catch-all row in the dict to keep it readable
demo_cols = [c for c in wide_export.columns if c.startswith(("turnout_2020_", "reg_2020_"))]
dictionary = pd.concat([
    dictionary,
    pd.DataFrame([(c, f"From geogiadata.org (2020 baseline)  -  see column name", "%") for c in demo_cols],
                 columns=["column", "description", "units"])
], ignore_index=True)

# Write both sheets
output_path = Path("ga_senate_2026_county_master.xlsx")  # saves alongside the notebook
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    wide_export.to_excel(writer, sheet_name="county_data", index=True)
    dictionary.to_excel(writer, sheet_name="data_dictionary", index=False)

print(f"\n[OK] Wrote {output_path.resolve()}")
print(f"  county_data:     {wide_export.shape[0]} rows  x  {wide_export.shape[1]} cols")
print(f"  data_dictionary: {len(dictionary)} entries")

Turnout tier distribution:
turnout_tier
Low     53
Mid     53
High    53
Name: count, dtype: int64

Tier  x  lean bucket cross-tab (count of counties):
turnout_tier  Low  Mid  High
lean_bucket                 
Strong R       26   30    38
Lean R         12    9     8
Competitive     4    7     3
Lean D          7    3     3
Strong D        4    4     1

[OK] Wrote C:\Users\langm\OneDrive\Documents\PROGRAMMING\GOVT520 - Campaign Management Institute\CMI Campaign Plan\ga_senate_2026_county_master.xlsx
  county_data:     159 rows  x  51 cols
  data_dictionary: 48 entries


## What the lean  x  turnout matrix actually tells us

Strategic implications cell by cell:

|              | Low turnout | Mid turnout | High turnout |
|--------------|-------------|-------------|--------------|
| Strong R     | **26 (GOTV gold)** | 30          | 38 (maintain)|
| Lean R       | 12          | 9           | 8            |
| Competitive  | 4           | **7 (sweet spot)** | 3        |
| Lean D       | 7 (defense) | 3           | 3            |
| Strong D     | 4           | 4           | 1            |

**The 26 Strong R + Low turnout counties are the single highest-leverage cell.**
These are counties where Collins's voters already exist but aren't showing up.
Every voter mobilized there is a near-guaranteed R vote. Mobilization dollars
go further here than anywhere else on the board.

**The 7 Competitive + Mid turnout counties are the persuasion sweet spot.**
Competitive enough that messaging can move the margin, mid-engaged enough that
voters will actually receive and process the message but you're not wasting
spend on people who'd vote anyway.

**Republican voters consistently show up; Democratic voters consistently don't.**
- Of 22 D-leaning counties (Lean D + Strong D), exactly half (11) are Low
  turnout  -  the D base under-mobilizes
- Of 123 R-leaning counties (Strong R + Lean R), only 38 are Low turnout
- That asymmetry is structural, not random. It's why R can win statewide
  despite D voter mass concentration in Atlanta

**The 4 Competitive + Low turnout counties are the dark horse.** Smallest cell
but highest upside-per-dollar  -  both persuadable AND under-engaged. Worth
zooming in on which 4 these are when you're in Sheets.

**The Strong D + High turnout county (just 1 county) is essentially Fulton or
DeKalb.** Don't touch  -  any turnout activity there nets D more than R.

## Polling data - loading and inspecting
NYT/538 export covers all states and races in long format (one row per candidate per question per poll). 
Need to filter to GA Senate 2026 general before anything else. First step: load the raw file and see the shape so 
I know what filters to apply.

File lives in a sibling folder to DATA_DIR (`Data\polling data\`), so navigating up one level with `.parent`.

In [21]:
polling_file = DATA_DIR.parent / "polling data" / "senate polling data from NYT 5.11.26.csv"
polls_raw = pd.read_csv(polling_file, low_memory=False)

print(f"Raw rows: {len(polls_raw):,}")
print(f"Columns ({len(polls_raw.columns)}):")
print(list(polls_raw.columns))

print(f"\nStates in the file: {len(polls_raw['state'].dropna().unique())}")
print(f"Cycles: {sorted(polls_raw['cycle'].dropna().unique())}")
print(f"\nStage breakdown:")
print(polls_raw['stage'].value_counts())

Raw rows: 1,891
Columns (48):
['poll_id', 'pollster_id', 'pollster', 'sponsor_ids', 'sponsors', 'display_name', 'pollster_rating_id', 'pollster_rating_name', 'numeric_grade', 'pollscore', 'methodology', 'transparency_score', 'state', 'start_date', 'end_date', 'sponsor_candidate_id', 'sponsor_candidate', 'sponsor_candidate_party', 'question_id', 'sample_size', 'population', 'subpopulation', 'population_full', 'tracking', 'created_at', 'notes', 'url', 'url_article', 'url_topline', 'url_crosstab', 'source', 'internal', 'partisan', 'cycle', 'office_type', 'seat_name', 'seat_number', 'election_date', 'stage', 'party', 'pct', 'answer', 'candidate_name', 'candidate_id', 'race_id', 'nationwide_match', 'ranked_choice_reallocated', 'hypothetical']

States in the file: 29
Cycles: [np.int64(2026)]

Stage breakdown:
stage
general           986
primary           867
primary runoff     38
Name: count, dtype: int64


## Filtering to GA Senate 2026 general
Three filters to apply: state == "GA", office_type == "U.S. Senate", stage == "general". Cycle is already all 
2026 so no filter needed there. Primary polls get dropped because they test R candidates against each other, 
not against Ossoff.

After filtering, looking at the unique candidates polled. The data is long format (one row per candidate per question), 
so this also tells me how many R candidates got tested in head-to-head against Ossoff - useful for the candidate-strength 
comp I want to do later (Collins vs other named Rs against the same Ossoff).

In [22]:
# Filter to GA Senate general only
ga_general = polls_raw[
    (polls_raw["state"] == "GA")
    & (polls_raw["office_type"] == "U.S. Senate")
    & (polls_raw["stage"] == "general")
].copy()

# Convert dates so I can sort/filter on them later
ga_general["start_date"] = pd.to_datetime(ga_general["start_date"], format="mixed", errors="coerce")
ga_general["end_date"] = pd.to_datetime(ga_general["end_date"], format="mixed", errors="coerce")

print(f"GA Senate 2026 general rows: {len(ga_general)}")
print(f"Unique poll-question combos: {ga_general[['poll_id', 'question_id']].drop_duplicates().shape[0]}")
print(f"Date range: {ga_general['end_date'].min().date()} to {ga_general['end_date'].max().date()}")

print(f"\nCandidates polled (and how many times):")
print(ga_general.groupby("candidate_name").size().sort_values(ascending=False))

GA Senate 2026 general rows: 136
Unique poll-question combos: 39
Date range: 2025-01-15 to 2026-04-09

Candidates polled (and how many times):
candidate_name
Don't know                39
Jon Ossoff                38
Someone else              14
Mike Collins               9
Buddy Carter               9
Derek Dooley               5
Would not vote             4
Brian Kemp                 4
Brad Raffensperger         4
John King                  4
Marjorie Taylor Greene     3
Generic Republican         1
Kelly Loeffler             1
Rich McCormick             1
dtype: int64


## Pivoting to Collins vs Ossoff head-to-head
Data is long - one row per candidate per question. I want one row per poll-question with collins_pct and 
ossoff_pct as columns, plus a margin column. Filtering to questions where BOTH candidates appear (skips any 
that only tested one). Also pulling other_pct so I can see how much support went to "Don't know" / "Someone else" 
/ third-party - useful signal for the size of the undecided universe.

Sorting by end_date descending so the most recent polls land at the top.

In [23]:
# Pivot from long to wide - one row per poll-question with Collins% and Ossoff%
results = []
for (pid, qid), group in ga_general.groupby(["poll_id", "question_id"]):
    collins_row = group[group["candidate_name"] == "Mike Collins"]
    ossoff_row  = group[group["candidate_name"] == "Jon Ossoff"]
    if collins_row.empty or ossoff_row.empty:
        continue
    info = group.iloc[0]
    other_pct = group[~group["candidate_name"].isin(["Mike Collins", "Jon Ossoff"])]["pct"].sum()
    results.append({
        "pollster":     info["pollster"],
        "sponsor":      info["sponsors"],
        "end_date":     info["end_date"].date(),
        "sample_size":  info["sample_size"],
        "population":   info["population"],
        "methodology":  info["methodology"],
        "partisan":     info["partisan"],
        "hypothetical": info["hypothetical"],
        "collins_pct":  float(collins_row["pct"].iloc[0]),
        "ossoff_pct":   float(ossoff_row["pct"].iloc[0]),
        "other_pct":    round(float(other_pct), 1),
        "margin":       round(float(collins_row["pct"].iloc[0]) - float(ossoff_row["pct"].iloc[0]), 1),
    })

collins_v_ossoff = pd.DataFrame(results).sort_values("end_date", ascending=False).reset_index(drop=True)
print(f"Collins vs Ossoff head-to-head polls: {len(collins_v_ossoff)}")
print(f"\nFull view (most recent first):")
print(collins_v_ossoff.to_string())

Collins vs Ossoff head-to-head polls: 8

Full view (most recent first):
           pollster                     sponsor    end_date  sample_size population                       methodology partisan  hypothetical  collins_pct  ossoff_pct  other_pct  margin
0  Echelon Insights                   NetChoice  2026-04-09        407.0         lv  Nonprobability Panel/Text-to-Web      NaN           NaN        44.00       51.00        5.0    -7.0
1   Emerson College                     Nexstar  2026-03-02       1000.0         lv  Text-to-Web/Nonprobability Panel      NaN           NaN        43.30       47.70        9.0    -4.4
2  Quantus Insights                         NaN  2025-09-12        624.0         rv  Nonprobability Panel/Text-to-Web      NaN           NaN        37.70       37.80       24.5    -0.1
3     TIPP Insights  League of American Workers  2025-08-01       2395.0         lv                               NaN      REP           NaN        44.10       44.94       11.0    -0.8
4  

## Reading the Collins vs Ossoff head-to-heads

Eight polls, Jan 2025 to April 2026. Every single one has Ossoff ahead. Margins range from -0.1 
(Quantus, Sep 2025) to -10 (WPA, Jan 2025). Most recent poll (Echelon, April 2026, n=407) shows -7.

Three patterns to flag:

- **R-sponsored polls can't find one where Collins leads either.** Three polls in this set are flagged 
  `partisan = REP` (TIPP/League of American Workers, Trafalgar, WPA/Club for Growth) plus Cygnal which leans R 
  in its house effects. All four still have Ossoff up. That's a floor signal - even the friendly pollsters 
  can't put Collins on top.

- **Big undecided universe in some polls.** Quantus shows 24.5% other/undecided with both candidates around 37%. 
  Most other polls show 9-21% undecided. Real persuasion room, especially if Collins-specific name ID is still building.

- **The trend is not in Collins's favor.** Earliest poll (WPA Jan 2025) had Ossoff +10. Most recent 
  (Echelon Apr 2026) has Ossoff +7. Middle polls range from -0.1 to -4.7. No clear trend toward Collins over 
  16 months, and the most recent number is worse than several from mid-2025.

Caveats:
- Echelon (most recent) is a small non-probability online panel (n=407) - wider error bars than the other polls
- Some polls tested multiple R candidates against Ossoff - the Collins-specific number is one of several scenarios
- House effects: WPA / Cygnal / Trafalgar / TIPP lean R; Emerson / Echelon / Quantus are more neutral

Implication for the writeup: partisan_blend_r is anchored on Walker's 2022 performance (~48.5% statewide). 
Polling shows Collins running BELOW that, which means the blend is probably too generous as a 2026 prediction. 
Worth flagging in the methodology section - the geographic targeting framework is sound, but the absolute 
partisan vote share it implies for Collins may be optimistic.

## What the lean  x  turnout matrix actually tells us

Strategic implications cell by cell:

|              | Low turnout | Mid turnout | High turnout |
|--------------|-------------|-------------|--------------|
| Strong R     | **26 (GOTV gold)** | 30          | 38 (maintain)|
| Lean R       | 12          | 9           | 8            |
| Competitive  | 4           | **7 (sweet spot)** | 3        |
| Lean D       | 7 (defense) | 3           | 3            |
| Strong D     | 4           | 4           | 1            |

**The 26 Strong R + Low turnout counties are the single highest-leverage cell.**
These are counties where Collins's voters already exist but aren't showing up.
Every voter mobilized there is a near-guaranteed R vote. Mobilization dollars
go further here than anywhere else on the board.

**The 7 Competitive + Mid turnout counties are the persuasion sweet spot.**
Competitive enough that messaging can move the margin, mid-engaged enough that
voters will actually receive and process the message but you're not wasting
spend on people who'd vote anyway.

**Republican voters consistently show up; Democratic voters consistently don't.**
- Of 22 D-leaning counties (Lean D + Strong D), exactly half (11) are Low
  turnout  -  the D base under-mobilizes
- Of 123 R-leaning counties (Strong R + Lean R), only 38 are Low turnout
- That asymmetry is structural, not random. It's why R can win statewide
  despite D voter mass concentration in Atlanta

**The 4 Competitive + Low turnout counties are the dark horse.** Smallest cell
but highest upside-per-dollar  -  both persuadable AND under-engaged. Worth
zooming in on which 4 these are when you're in Sheets.

**The Strong D + High turnout county (just 1 county) is essentially Fulton or
DeKalb.** Don't touch  -  any turnout activity there nets D more than R.

## Polling summary stats + comparison to partisan blend
Four numbers worth pulling out: simple average across all 8 polls, sample-size-weighted average (gives bigger 
polls more influence), recent-only average (last 12 months), and 2026-only average. Then comparing the weighted 
polling margin to what my partisan_blend_r implies statewide.

Why compare to the blend: blend is anchored on 2022 Sen (Walker), which got 48.5%. If polls have Collins at 
something different, the gap tells me whether the 2026 environment is shifting away from the 2022 baseline. 
This is the empirical check on whether my targeting framework's assumed partisan share is realistic.

In [24]:
polls = collins_v_ossoff.copy()
polls["end_date"] = pd.to_datetime(polls["end_date"])

# Simple and weighted averages across all 8 polls
simple_avg   = polls["margin"].mean()
weighted_avg = (polls["margin"] * polls["sample_size"]).sum() / polls["sample_size"].sum()
print(f"All 8 polls:")
print(f"  Simple average margin:   {simple_avg:+.2f} pts")
print(f"  Weighted average margin: {weighted_avg:+.2f} pts")

# Last 12 months only
recent = polls[polls["end_date"] >= "2025-05-12"]
recent_simple   = recent["margin"].mean()
recent_weighted = (recent["margin"] * recent["sample_size"]).sum() / recent["sample_size"].sum()
print(f"\nLast 12 months only (n={len(recent)}):")
print(f"  Simple average:    {recent_simple:+.2f} pts")
print(f"  Weighted average:  {recent_weighted:+.2f} pts")

# 2026 polls only (just 2 polls)
twenty_six = polls[polls["end_date"] >= "2026-01-01"]
print(f"\n2026 polls only (n={len(twenty_six)}):")
print(f"  Simple average:    {twenty_six['margin'].mean():+.2f} pts")

# Compare to partisan blend - vote-weighted statewide implied R share
blend_implied_r      = (wide["partisan_blend_r"] * wide["active_voters_2026"]).sum() / wide["active_voters_2026"].sum()
blend_implied_margin = 2 * (blend_implied_r - 50)
print(f"\nPartisan blend statewide implied:")
print(f"  Vote-weighted blend R share:   {blend_implied_r:.2f}%")
print(f"  Implied margin (2*(blend-50)): {blend_implied_margin:+.2f} pts")

print(f"\nGap (polling weighted - blend implied): {weighted_avg - blend_implied_margin:+.2f} pts")

All 8 polls:
  Simple average margin:   -4.04 pts
  Weighted average margin: -3.11 pts

Last 12 months only (n=6):
  Simple average:    -2.93 pts
  Weighted average:  -2.41 pts

2026 polls only (n=2):
  Simple average:    -5.70 pts

Partisan blend statewide implied:
  Vote-weighted blend R share:   49.35%
  Implied margin (2*(blend-50)): -1.31 pts

Gap (polling weighted - blend implied): -1.80 pts


## Reading the polling summary stats

Three findings:

- **The blend is about 1.5-2 pts too generous to Collins.** Vote-weighted partisan blend implies Collins 
  at 49.35% statewide (margin -1.31). Polling weighted average says -3.11. Gap = -1.80. Same direction 
  across all the slicings - polls consistently show Collins doing worse than 2022 Walker's geography 
  would predict.

- **Recent polls are slightly less bad than 2025.** Last-12-months weighted average is -2.41 vs -3.11 for 
  the full sample. The Jan 2025 WPA -10 is dragging the full set down; without it the race looks closer. 
  Reasonable to use the last-12-months number (-2.41) as the more current benchmark.

- **But 2026 polls specifically are the worst.** Just 2 polls so this is noisy, but Echelon (-7) and 
  Emerson (-4.4) average -5.7. Either the race is widening as we get closer to the election, or these 
  two polls are outliers. Hard to tell with n=2.

What this means for the campaign plan:
- Treat partisan_blend_r as a CEILING for what's realistically achievable, not a baseline. The 49.35% 
  statewide blend is the "if Collins runs as well as Walker did in his geography" number. Polling 
  suggests Collins is currently running 1-2 pts below that.
- Closing the gap to the blend (and beyond) is what the persuasion universe is for. The Kemp-Walker 
  gap counties identified earlier in the targeting work are exactly where this gap could close.
- In the writeup methodology section, note this explicitly: the blend is an optimistic anchor; 
  polling is the realistic floor.

## House effects - are R-leaning polls showing a different race?
Four of the 8 polls are R-sponsored or have R-leaning house effects (TIPP, Trafalgar, WPA, Cygnal). 
Conventional wisdom is R-sponsored polls show R-favorable margins. Splitting the data to check whether 
the R subset is meaningfully tighter than the neutral subset, or whether the gap is consistent across.

If R-leaning polls show Collins behind by less, the "true" race is probably even more Ossoff-favorable 
than the weighted average suggests. If both subsets show the same gap, polling is consistent and I can 
trust the overall weighted average.

In [25]:
# R-leaning = flagged partisan REP OR pollster has known R house effect
r_leaning_pollsters = {"TIPP Insights", "Trafalgar Group", "WPA Intelligence", "Cygnal Political"}
polls["r_leaning"] = polls["partisan"].eq("REP") | polls["pollster"].isin(r_leaning_pollsters)

for label, subset in [("R-leaning house", polls[polls["r_leaning"]]),
                      ("Neutral",         polls[~polls["r_leaning"]])]:
    n = len(subset)
    if n == 0:
        print(f"{label:<18}  n=0")
        continue
    avg = subset["margin"].mean()
    weighted = (subset["margin"] * subset["sample_size"]).sum() / subset["sample_size"].sum()
    print(f"{label:<18}  n={n}  simple avg = {avg:+.2f}  weighted = {weighted:+.2f}")

print(f"\nFor reference: blend-implied margin = -1.31 pts")
print(f"\nIndividual polls by group:")
print(polls.sort_values("r_leaning")[["pollster", "end_date", "margin", "r_leaning"]].to_string(index=False))

R-leaning house     n=5  simple avg = -4.16  weighted = -2.99
Neutral             n=3  simple avg = -3.83  weighted = -3.60

For reference: blend-implied margin = -1.31 pts

Individual polls by group:
        pollster   end_date  margin  r_leaning
Echelon Insights 2026-04-09    -7.0      False
 Emerson College 2026-03-02    -4.4      False
Quantus Insights 2025-09-12    -0.1      False
   TIPP Insights 2025-08-01    -0.8       True
   TIPP Insights 2025-08-01    -2.9       True
Cygnal Political 2025-05-17    -2.4       True
 Trafalgar Group 2025-04-27    -4.7       True
WPA Intelligence 2025-01-15   -10.0       True


## Reading the house effects split

Both subsets land in the same range:
- R-leaning polls (n=5): simple -4.16, weighted -2.99
- Neutral polls (n=3): simple -3.83, weighted -3.60

This is actually good methodologically. The conventional house-effect concern is that R-sponsored polls 
systematically show tighter races than neutral ones - if that were happening here, the R-leaning weighted 
average would be much closer to zero than the neutral one. Instead they're within 0.6 pts of each other 
on the weighted average, and the R subset is slightly WORSE on simple average (driven by WPA's -10).

Takeaway: polling is consistent across house effects. I can trust the overall weighted average (-3.11) 
as a clean read on the race rather than worrying that R-sponsored polls are pulling the picture in a 
particular direction.

The WPA Jan 2025 poll (-10) is the one notable outlier - much worse than even the other R-leaning polls. 
Could be early-cycle noise or a methodological quirk. Worth dropping when computing the "current" weighted 
average since it's also the oldest.

## Candidate strength - Collins vs other Rs against the same Ossoff
The NYT file tested 10 different R candidates against Ossoff across 39 poll-questions. Most useful for me 
is comparing Collins's average margin against the same opponent (Ossoff) to what other R candidates would 
do. If Collins underperforms a generic-R alternative, that's a candidate-quality signal - and it means the 
partisan_blend_r ceiling is probably even more generous than my polling-vs-blend gap already suggests.

Two views: 
1. Each R candidate's avg margin across all the polls they appeared in (simple averaging)
2. Same-poll comparison - where multiple Rs appear in the same poll, side-by-side margins

In [26]:
r_candidates = ["Mike Collins", "Buddy Carter", "Derek Dooley", "Brian Kemp",
                "Brad Raffensperger", "John King", "Marjorie Taylor Greene",
                "Generic Republican", "Kelly Loeffler", "Rich McCormick"]

results = []
for (pid, qid), group in ga_general.groupby(["poll_id", "question_id"]):
    ossoff_row = group[group["candidate_name"] == "Jon Ossoff"]
    if ossoff_row.empty:
        continue
    r_rows = group[group["candidate_name"].isin(r_candidates)]
    if r_rows.empty:
        continue
    info = group.iloc[0]
    for _, r_row in r_rows.iterrows():
        results.append({
            "r_candidate": r_row["candidate_name"],
            "pollster":    info["pollster"],
            "end_date":    info["end_date"].date(),
            "sample_size": info["sample_size"],
            "r_pct":       float(r_row["pct"]),
            "ossoff_pct":  float(ossoff_row["pct"].iloc[0]),
            "margin":      round(float(r_row["pct"]) - float(ossoff_row["pct"].iloc[0]), 1),
        })

comp = pd.DataFrame(results)

# Each R candidate aggregated
summary = comp.groupby("r_candidate").apply(
    lambda g: pd.Series({
        "n_polls":    len(g),
        "simple_avg": round(g["margin"].mean(), 2),
        "weighted":   round((g["margin"] * g["sample_size"]).sum() / g["sample_size"].sum(), 2),
    })
).sort_values("simple_avg", ascending=False)

print("Each R candidate's avg margin vs Ossoff:\n")
print(summary)

# Same-poll comparison - pivot so each row is one poll, columns are R candidates
print("\nSame-poll comparison (margin vs Ossoff by candidate):")
pivot = comp.pivot_table(
    index=["pollster", "end_date"],
    columns="r_candidate",
    values="margin",
    aggfunc="first",
)
print(pivot.to_string())

Each R candidate's avg margin vs Ossoff:

                        n_polls  simple_avg  weighted
r_candidate                                          
Brian Kemp                  4.0        5.03      4.67
Generic Republican          1.0        0.20      0.20
Kelly Loeffler              1.0       -3.50     -3.50
Mike Collins                8.0       -4.04     -3.11
Derek Dooley                4.0       -5.52     -4.77
Buddy Carter                8.0       -5.66     -3.98
Brad Raffensperger          4.0       -8.07     -7.82
John King                   4.0      -10.28     -9.88
Rich McCormick              1.0      -11.00    -11.00
Marjorie Taylor Greene      3.0      -13.57    -13.53

Same-poll comparison (margin vs Ossoff by candidate):
r_candidate                       Brad Raffensperger  Brian Kemp  Buddy Carter  Derek Dooley  Generic Republican  John King  Kelly Loeffler  Marjorie Taylor Greene  Mike Collins  Rich McCormick
pollster              end_date                               

## Reading the candidate strength comparison

Key findings:

- **Kemp is the only R who beats Ossoff.** Average +5 across 4 polls (Quantus 2/13, Tyson Group 1/31, 
  UGA 4/24, WPA 1/15). Confirms what the Kemp-Walker gap already showed - Kemp's brand is the ceiling 
  for any GA Republican. Not a realistic 2026 candidate since he's term-limited, but useful as the 
  "what's possible for an R in this state" benchmark.

- **Collins is mid-pack among realistic alternatives, not a candidate-quality outlier.** He averages -3.11 
  weighted, which puts him roughly tied with Carter (-3.98 weighted) and ahead of Dooley (-4.77), 
  Raffensperger (-7.82), and well ahead of MTG (-13.53). Same-poll comparisons confirm this:
  - Quantus 9/12/25: Collins -0.1, Carter -3.1, Dooley -6.7 (Collins best by 3+ pts)
  - TIPP 8/1/25: Collins -0.8, Carter -1.7, Dooley -3.2 (Collins best)
  - Trafalgar 4/27/25: Collins -4.7, Raffensperger -8.9, MTG -11.8 (Collins best)
  - Echelon 4/9/26: Collins -7.0, Carter -9.0 (Collins better by 2)
  - Emerson 3/2/26: Collins -4.4, Carter -3.4, Dooley -7.5 (Carter slightly better, Collins better than Dooley)

- **MTG is a disaster candidate** (-13.57 avg). Confirms the polarization story - the further you push 
  toward a movement-conservative candidate the more Ossoff wins.

- **Raffensperger tests surprisingly poorly** (-8.07) despite high name ID. Suggests Trump-aligned 
  Republican primary voters' distaste for him (post-2020) bleeds into general election performance.

Implication for the campaign plan:
- Collins doesn't have a candidate-quality penalty relative to his peers. The gap between his polling and 
  the Kemp ceiling (~8-10 pts) is the gap between any Trump-aligned R and a popular incumbent governor brand - 
  NOT a Collins-specific weakness.
- This validates using Walker 2022 as the partisan_blend_r anchor. Collins is running about where you'd 
  expect a Trump-aligned R to run, which is what Walker was.
- The path to victory is the same one any Trump-aligned R would need: close the Kemp-Walker gap by winning 
  back the suburban voters who pulled Kemp but not Walker. The targeting work already identified those 
  counties (Oconee, Forsyth, Cobb, Cherokee, Hall, Fayette).

## Building the targeting matrix columns
Four columns to add to the wide table so I can hand off a per-county action plan, not just a partisan score:

1. `targeting_action` - the recommended action per county based on (lean_bucket, turnout_tier) combo. 
   This is the matrix in code form - "Strong R + Low turnout = MOBILIZE", "Competitive + any = PERSUADE", 
   etc. One label per county, makes the spreadsheet directly usable.

2. `persuasion_universe` - active_voters × kemp_walker_gap% per county. The estimated size of the 
   split-ticket pool that picked Kemp but not Walker - the voters Collins needs to win back. Cap at 0 
   for counties where Walker actually outperformed Kemp.

3. `mobilization_universe` - active_voters minus proj_2026_method2_votes. The voters not projected to 
   show up. In R-leaning counties this is R voters Collins can mobilize; in D-leaning counties it's 
   D voters we WANT to stay home. Cap at 0.

4. `priority_score` and `priority_rank` - composite score weighted by lean bucket strategic value, 
   then ranked 1-159. In R-leaning counties mobilization dominates the score; in Competitive counties 
   persuasion dominates; in Lean D it's defensive only; Strong D gets zero. priority_rank 1 = top 
   spending priority.

The point of priority_rank: turns the matrix into an ACTIONABLE list. "Go spend in these 20 counties 
first" is more useful than "here are 159 counties with various scores."

In [27]:
# 1. Targeting action lookup based on (lean_bucket, turnout_tier)
def targeting_action(lean, tier):
    if lean == "Strong R":
        return "MAINTAIN" if tier == "High" else "MOBILIZE"
    if lean == "Lean R":
        return "MOBILIZE + LIGHT PERSUADE"
    if lean == "Competitive":
        return "PERSUADE"
    if lean == "Lean D":
        return "DEFENSE / MIN INVEST" if tier == "Low" else "DEFENSE"
    return "SKIP"  # Strong D

wide["targeting_action"] = wide.apply(
    lambda r: targeting_action(r["lean_bucket"], r["turnout_tier"]), axis=1
)

# 2. Persuasion universe - active voters × Kemp-Walker gap, clipped at 0
wide["persuasion_universe"] = (
    wide["active_voters_2026"] * wide["kemp_walker_gap_pts"].clip(lower=0) / 100
).round(0).astype(int)

# 3. Mobilization universe - active voters not projected to vote, clipped at 0
wide["mobilization_universe"] = (
    (wide["active_voters_2026"] - wide["proj_2026_method2_votes"]).clip(lower=0)
).round(0).astype(int)

# 4. Priority score - weighted by lean bucket strategic value
def priority_score(row):
    if row["lean_bucket"] in ("Strong R", "Lean R"):
        return row["mobilization_universe"] + 0.5 * row["persuasion_universe"]
    if row["lean_bucket"] == "Competitive":
        return 1.5 * row["persuasion_universe"] + row["mobilization_universe"]
    if row["lean_bucket"] == "Lean D":
        return 0.3 * row["persuasion_universe"]
    return 0  # Strong D

wide["priority_score"] = wide.apply(priority_score, axis=1).round(0).astype(int)
wide["priority_rank"]  = wide["priority_score"].rank(ascending=False, method="min").astype(int)

print(f"Wide table now has {wide.shape[1]} columns")

print(f"\nTargeting action distribution:")
print(wide["targeting_action"].value_counts())

print(f"\nTop 20 counties by priority rank:")
cols = ["lean_bucket", "turnout_tier", "targeting_action", "active_voters_2026",
        "persuasion_universe", "mobilization_universe", "priority_score", "priority_rank"]
print(wide.sort_values("priority_rank")[cols].head(20).to_string())

Wide table now has 56 columns

Targeting action distribution:
targeting_action
MOBILIZE                     56
MAINTAIN                     38
MOBILIZE + LIGHT PERSUADE    29
PERSUADE                     14
SKIP                          9
DEFENSE / MIN INVEST          7
DEFENSE                       6
Name: count, dtype: int64

Top 20 counties by priority rank:
           lean_bucket turnout_tier           targeting_action  active_voters_2026  persuasion_universe  mobilization_universe  priority_score  priority_rank
county                                                                                                                                                       
Cherokee      Strong R         High                   MAINTAIN              205421                13660                  71315           78145              1
Forsyth       Strong R         High                   MAINTAIN              170617                12865                  61736           68168              2
Hall

## Reading the priority county list

Top 20 is dominated by R-leaning counties with big voter mass - mostly the suburban Atlanta arc 
(Cherokee, Forsyth, Hall, Paulding, Coweta) plus smaller Strong R / Lean R counties (Houston, 
Columbia, Bartow, Carroll, Lowndes) where the mobilization gap is big.

Three things to flag:

- **The top 3 priorities (Cherokee, Forsyth, Hall) are GOTV plays, not persuasion plays.** Cherokee 
  ranks #1 despite being MAINTAIN (Strong R + High turnout) - the priority score is high purely because 
  of voter mass. These three counties combined have ~35K persuasion universe and ~195K mobilization 
  universe. The big number is the mobilization.

- **Only Fayette makes the top 20 from the Competitive bucket** (rank 11, PERSUADE). The persuasion 
  universe is structurally smaller than the mobilization universe - Kemp-Walker gaps are 4-7 pts max, 
  while mobilization gaps in Strong R counties are often 20-30 pts of active voters. Translation: 
  turnout wins more votes than persuasion in this state.

- **Cobb / Gwinnett / Henry are missing from the top 20** despite having huge persuasion universes. 
  They're Lean D bucket, which gets weighted 0.3× in the priority score because Collins isn't going to 
  win those counties outright. That's defensible (mobilizing D-base counties helps Ossoff more than 
  Collins) but worth noting: Cobb specifically has the biggest absolute Kemp-Walker gap in the state 
  in raw vote terms (~35K voters). A more aggressive persuasion-first strategy could reweight them 
  upward - this is a strategic choice, not a data finding.

Strategic implication: Collins's path is more about TURNOUT than persuasion. Maximizing R base turnout 
in 30-50 counties produces more votes than persuading in a handful of competitive ones. The Competitive 
bucket isn't a major source of votes per county - it's where the marginal seats are ACTUALLY won/lost 
given consistent base turnout. So the budget should go HEAVY in Strong R / Lean R counties to mobilize, 
and SURGICAL in the 14 Competitive counties (especially Competitive + Mid/High turnout).

Action distribution: 56 MOBILIZE, 38 MAINTAIN, 29 MOBILIZE + LIGHT PERSUADE, 14 PERSUADE, 13 DEFENSE 
variants, 9 SKIP. This breakdown is what the budget allocation should map to next.

## Targeting matrix - aggregate view by cell
The matrix as it'll appear in the campaign plan: for each (lean_bucket, turnout_tier) cell, show 
counties, voters, persuasion universe, and mobilization universe. Lets the team see at a glance where 
the volume is and what kind of campaign action that cell deserves.

In [28]:
# Aggregate metrics by matrix cell
cell_agg = wide.groupby(["lean_bucket", "turnout_tier"], observed=True).agg(
    counties=("active_voters_2026", "size"),
    active_voters=("active_voters_2026", "sum"),
    persuasion_universe=("persuasion_universe", "sum"),
    mobilization_universe=("mobilization_universe", "sum"),
).astype(int)

# Reorder rows/columns
lean_order = ["Strong R", "Lean R", "Competitive", "Lean D", "Strong D"]
tier_order = ["Low", "Mid", "High"]
cell_agg = cell_agg.reindex(pd.MultiIndex.from_product([lean_order, tier_order],
                                                       names=["lean_bucket", "turnout_tier"]),
                            fill_value=0)

print("MATRIX CELL SUMMARY - one row per (lean, turnout) combination:\n")
print(cell_agg.to_string())

# Cross-tab views for the writeup
print(f"\n\nCounties per cell:")
print(pd.crosstab(wide["lean_bucket"], wide["turnout_tier"]).reindex(lean_order)[tier_order])

print(f"\nActive voters per cell:")
voters_xtab = wide.pivot_table(index="lean_bucket", columns="turnout_tier",
                                values="active_voters_2026", aggfunc="sum", fill_value=0)
print(voters_xtab.reindex(lean_order)[tier_order].astype(int))

print(f"\nPersuasion universe per cell:")
pers_xtab = wide.pivot_table(index="lean_bucket", columns="turnout_tier",
                              values="persuasion_universe", aggfunc="sum", fill_value=0)
print(pers_xtab.reindex(lean_order)[tier_order].astype(int))

print(f"\nMobilization universe per cell:")
mob_xtab = wide.pivot_table(index="lean_bucket", columns="turnout_tier",
                             values="mobilization_universe", aggfunc="sum", fill_value=0)
print(mob_xtab.reindex(lean_order)[tier_order].astype(int))

MATRIX CELL SUMMARY - one row per (lean, turnout) combination:

                          counties  active_voters  persuasion_universe  mobilization_universe
lean_bucket turnout_tier                                                                     
Strong R    Low                 26         597116                25555                 286357
            Mid                 30         722179                33664                 316396
            High                38        1196695                66611                 429515
Lean R      Low                 12         271767                 9145                 135880
            Mid                  9         429048                17984                 189856
            High                 8         161818                 7511                  61582
Competitive Low                  4          43753                 1278                  21074
            Mid                  7          73382                 2323                  31

## Reading the matrix cell view

The matrix in four flavors lets me see where each kind of opportunity actually sits. Five findings worth 
putting in the writeup:

**1. The single biggest cell by voter mass is Strong R + High turnout (1.2M voters, 38 counties).** 
   Action is MAINTAIN - these voters already show up and already vote R. Don't over-spend here, but 
   don't take them for granted either. The persuasion universe inside this cell (67K) is actually the 
   biggest of any cell - more on that below.

**2. The GOTV target cell is Strong R + Low/Mid turnout: 56 counties, 1.32M voters, 600K mobilization 
   universe.** This is where the per-dollar ROI on mobilization is highest. Voters Collins wins when 
   they show up; they just haven't been showing up.

**3. The persuasion-universe story is counterintuitive: it's biggest INSIDE Strong R counties.**
   - Strong R: 126K persuasion universe (35% of state total)
   - Lean D: 101K (28%)
   - Strong D: 85K (24%)
   - Lean R: 35K (10%)
   - Competitive: 10K (3%)
   
   The COMPETITIVE bucket has the smallest persuasion universe in absolute terms (only 10K) because 
   only 228K voters live there total. Translation: the "persuasion play" Collins needs to make is 
   primarily inside counties he already wins - winning back suburbanites in Cherokee / Forsyth / 
   Hall / Coweta who pulled Kemp but not Walker. The Competitive bucket matters for COMPLETENESS but 
   it's not where the volume is.

**4. The defensive cell to watch is Lean D + Low turnout: 7 counties, 1.17M voters, 578K mobilization 
   universe.** These are D-base counties with massive under-mobilization (Bibb, Chatham, Richmond, 
   Dougherty area). Action is DEFENSE / MIN INVEST - don't spend turnout dollars there because 
   mobilizing D voters helps Ossoff. The 578K mobilization universe here is a sleeping giant for 
   the D side; Ossoff's job is to wake them up; Collins's job is to make sure they stay home.

**5. Statewide universe totals put the race in scale.** ~357K total persuasion universe, ~3.2M total 
   mobilization universe. Walker lost by 37K in 2022. Even converting 10% of the persuasion pool 
   would more than cover that margin. The math is winnable - the question is whether Collins can 
   execute the conversion.

For the campaign plan budget allocation: spend in proportion to (persuasion universe + mobilization 
universe) per cell, weighted by action cell. Highest leverage = Strong R Low/Mid + Lean R + 
suburban-Atlanta corners of the Strong R bucket. Lowest leverage = Strong D + the highest-turnout 
Lean D cells.

## Updating data dictionary + re-exporting xlsx
Added 4 new columns to the wide table (targeting_action, persuasion_universe, mobilization_universe, 
priority_score, priority_rank). Adding them to the dictionary and re-running the export so the team's 
copy of ga_senate_2026_county_master.xlsx has the actionable per-county view.

In [29]:
# Add the new columns to the data dictionary
new_entries = pd.DataFrame([
    ("targeting_action",      "Recommended action per county from lean × turnout cell (MOBILIZE / MAINTAIN / PERSUADE / DEFENSE / SKIP)", "category"),
    ("persuasion_universe",   "active_voters × kemp_walker_gap% - estimated split-ticket Kemp-not-Walker pool", "voters"),
    ("mobilization_universe", "active_voters minus proj_2026_method2_votes - voters not projected to turn out", "voters"),
    ("priority_score",        "Composite opportunity score weighted by lean bucket strategic value", "score"),
    ("priority_rank",         "1 to 159 ranking on priority_score - rank 1 = highest spending priority", "rank"),
], columns=["column", "description", "units"])

dictionary = pd.concat([dictionary, new_entries], ignore_index=True)
print(f"Data dictionary now has {len(dictionary)} entries")

# Re-sort wide_export with the new columns and re-export
wide_export = wide.copy()
wide_export["lean_bucket"] = wide_export["lean_bucket"].astype(bucket_order)
wide_export = wide_export.sort_values(
    ["lean_bucket", "active_voters_2026"], ascending=[True, False]
)

output_path = Path("ga_senate_2026_county_master.xlsx")
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    wide_export.to_excel(writer, sheet_name="county_data", index=True)
    dictionary.to_excel(writer, sheet_name="data_dictionary", index=False)

print(f"\n[OK] Wrote {output_path.resolve()}")
print(f"  county_data:     {wide_export.shape[0]} rows × {wide_export.shape[1]} cols")
print(f"  data_dictionary: {len(dictionary)} entries")

Data dictionary now has 53 entries

[OK] Wrote C:\Users\langm\OneDrive\Documents\PROGRAMMING\GOVT520 - Campaign Management Institute\CMI Campaign Plan\ga_senate_2026_county_master.xlsx
  county_data:     159 rows × 56 cols
  data_dictionary: 53 entries


## Win number and path to victory
Quick math to anchor the strategy: what's the actual vote target for Collins to win, and how does he get 
there from the county-level projections?

GA's 50%+1 runoff rule means Collins needs >50% to win outright in November. Otherwise December runoff. 
Two turnout scenarios give two win numbers:

- Method 1 (conservative): ~3.94M total votes -> need ~1.97M to clear 50%
- Method 2 (expected): ~4.14M total votes -> need ~2.07M to clear 50%

Then compare against expected R votes if Collins matches the partisan_blend_r per county. The gap is 
what the persuasion + mobilization universe needs to close.

In [30]:
# Win thresholds
m1_total = wide["proj_2026_method1_votes"].sum()
m2_total = wide["proj_2026_method2_votes"].sum()
win_m1 = int(m1_total * 0.5) + 1  # 50%+1 vote
win_m2 = int(m2_total * 0.5) + 1

# Expected R votes per county if Collins matches the partisan_blend_r
wide["expected_r_votes_m1"] = (wide["proj_2026_method1_votes"] * wide["partisan_blend_r"] / 100).round(0).astype(int)
wide["expected_r_votes_m2"] = (wide["proj_2026_method2_votes"] * wide["partisan_blend_r"] / 100).round(0).astype(int)

exp_r_m1 = wide["expected_r_votes_m1"].sum()
exp_r_m2 = wide["expected_r_votes_m2"].sum()

print("STATEWIDE TARGETS\n")
print(f"{'':40} {'Method 1':>15} {'Method 2':>15}")
print(f"{'-'*70}")
print(f"{'Projected total turnout':40} {m1_total:>15,} {m2_total:>15,}")
print(f"{'Win number (50%+1)':40} {win_m1:>15,} {win_m2:>15,}")
print(f"{'Expected R votes (Collins at blend)':40} {exp_r_m1:>15,} {exp_r_m2:>15,}")
print(f"{'Gap to win number':40} {exp_r_m1 - win_m1:>+15,} {exp_r_m2 - win_m2:>+15,}")
print(f"{'Expected R share':40} {exp_r_m1/m1_total*100:>14.2f}% {exp_r_m2/m2_total*100:>14.2f}%")

print(f"\n\nR VOTE CONTRIBUTION BY LEAN BUCKET (Method 2)\n")
order = ["Strong R", "Lean R", "Competitive", "Lean D", "Strong D"]
bucket = wide.groupby("lean_bucket").agg(
    counties=("active_voters_2026", "size"),
    projected_votes=("proj_2026_method2_votes", "sum"),
    expected_r_votes=("expected_r_votes_m2", "sum"),
).reindex(order)
bucket["% of total R votes"] = (bucket["expected_r_votes"] / bucket["expected_r_votes"].sum() * 100).round(1)
print(bucket.to_string())

STATEWIDE TARGETS

                                                Method 1        Method 2
----------------------------------------------------------------------
Projected total turnout                        3,935,924       4,139,248
Win number (50%+1)                             1,967,963       2,069,625
Expected R votes (Collins at blend)            1,934,002       2,068,521
Gap to win number                                -33,961          -1,104
Expected R share                                  49.14%          49.97%


R VOTE CONTRIBUTION BY LEAN BUCKET (Method 2)

             counties  projected_votes  expected_r_votes  % of total R votes
lean_bucket                                                                 
Strong R           94          1483722           1077991                52.1
Lean R             29           475315            286850                13.9
Competitive        14           137709             69741                 3.4
Lean D             13          1016969

## Reading the win number results

Three findings to anchor the strategy:

- **Collins is essentially tied at the partisan blend.** Under Method 2 (expected turnout) the gap is just 
  -1,104 votes - a statistical tie. Under Method 1 (flat 2022 baseline) the gap is -33,961, which is 
  almost exactly Walker's 2022 loss margin (~37K). The race is winnable but doesn't win itself - the 
  blend gives Collins a ceiling around 50%, and he needs to either match the blend perfectly across the 
  state OR close the gap with persuasion/mobilization on top.

- **The persuasion universe (~360K) is way more than the gap (-1K to -34K).** Converting even 10% of 
  the Kemp-Walker split-ticket pool closes the worst-case Method 1 deficit and produces a winning margin. 
  That's the empirical justification for the suburban Atlanta persuasion strategy - the math works.

- **Lean D counties produce MORE R votes than Lean R counties.** 19.4% of expected R votes come from 
  Lean D bucket (Cobb, Gwinnett, Henry, etc.) vs only 13.9% from Lean R. This is purely a voter-mass 
  story: Lean D counties have 5x more total voters than Lean R counties, so even at 40% R support they 
  produce more raw R votes. Implication: the "ignore Lean D entirely" stance from the targeting matrix 
  is too aggressive. Defensive persuasion in Cobb specifically (biggest Kemp-Walker gap in raw votes) 
  is worth doing - not to win the county, but because the R voters there are real and they affect the 
  statewide total.

Where the votes come from (Method 2 expected R total: 2.07M):
- Strong R counties: 52.1% (1.08M votes) - the largest single contributor
- Lean D counties: 19.4% (402K) - second-largest,